# 02 - Retrieve, Refine Candidates (API) - AIC 2026

Notebook 2/3. Input là artifact của NB01 (`artifact_manifest.json`); output là **review package** cho NB03.

1. Load FAISS Visual, FAISS Text, BM25, Object index, docs và keyframes.
2. Parse query: giữ nguyên `q_vi`, tạo `q_en` bằng MiMo có kiểm soát và sinh structured schema có validate.
3. Retrieval theo các nhánh SigLIP2, text embedding, BM25, video-level summary/metadata và Object soft boost.
4. Weighted RRF fusion, video prior, temporal NMS và diversity.
5. Voyage Rerank Text trên Top-N.
6. Dense frame refinement trên video gốc, với coarse-to-fine và rescore SigLIP2.
7. Nhánh riêng cho KIS, Q&A và TRAKE.
8. Xuất tối đa 100 candidate/query, contact sheet và `review_manifest.json`.

Notebook này không làm human review và có thể chạy lại mà không gọi API. Mỗi query ghi checkpoint riêng trong `candidates/<query_id>.json`.

In [ ]:
# ============================== CELL 0: DEPENDENCIES ==============================
# Kaggle image Không có sẵn FAISS -> Notebook tự cai. Cần Settings -> Internet = On.
# CELL này chạy được cả ở chế độ Save & Run All (Commit), không phụ thuộc thao tác tay.
import importlib, subprocess, sys

def ensure(module, pip_name=None, upgrade=False):
    """Import được thì thôi; không thì pip install rồi Import lại. Trả về True/False."""
    if not upgrade:
        try:
            importlib.import_module(module)
            print("  OK     ", module)
            return True
        except Exception:
            pass
    pkg = pip_name or module
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + (["-U"] if upgrade else []) + [pkg]
    print("  cai    ", pkg, "...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("  [Lỗi] không cai được", pkg, "->", (r.stderr or r.stdout or "")[-400:])
        return False
    importlib.invalidate_caches()
    try:
        importlib.import_module(module)
        print("  OK     ", module, "(vừa cai)")
        return True
    except Exception as e:
        print("  [Lỗi] cai xong vẫn Import lỗi:", e)
        return False

print("kiểm tra thư viện:")
FAISS_OK = ensure("faiss", "faiss-cpu")
for _m in ["pandas", "pyarrow", "scipy", "numpy"]:
    ensure(_m)
for _m, _p in [("torch", None), ("cv2", "OpenCV-python-headless"), ("PIL", "Pillow")]:
    ensure(_m, _p)
try:
    import transformers
    _v = tuple(int(x) for x in transformers.__version__.split(".")[:2])
    print("  transformers", transformers.__version__)
    if _v < (4, 49):
        print("  -> đã có checkpoint cho SigLIP2, đang nâng cấp...")
        ensure("transformers", "transformers", upgrade=True)
        print("  [CHÚ Ý] nâng cấp xong PHẢI Restart Kernel rồi chạy lại từ đầu.")
except Exception:
    ensure("transformers", "transformers", upgrade=True)

if not FAISS_OK:
    print("")
    print("[Cảnh báo] Thiếu FAISS -> không Build/đọc được FAISS Index.")
    print("  1) Settings -> Internet = On")
    print("  2) Chạy lại đúng CELL này")
    print("  3) Nếu vẫn lỗi: thêm một cell và chạy  !pip install -q faiss-cpu")


In [ ]:
# ============================== CELL 1: CONFIG ==============================
import os, sys, json, math, re, time, base64, hashlib, unicodedata, gc, io, shutil
from pathlib import Path
import numpy as np, pandas as pd

CFG = {
    "art_dir":    "/kaggle/working/artifacts",       # resolved NB01 Output dir
    "art_zip":    "/kaggle/input/notebooks/kitnehi1211/01-build-indices-api/_output_.zip",
    "cache_dir":  "/kaggle/temp/nb02_cache",
    "out_dir":    "/kaggle/working/review_package",  # bàn giao NB03
    # Layout thật trên Kaggle của bạn. Để None -> lấy từ manifest NB01 / Auto-discover.
    "query_dir":  "/kaggle/input/datasets/kitnehi1211/dethithunghiem",
    "query_scan_depth": 5,
    "only_queries": [],          # ["Query-p1-1-KIS"] để chạy 1 Query
    "force_requery": False,      # True -> bỏ qua Checkpoint per-Query

    # --- Top-k từng retriever (baseline) ---
    "topk": {
        "Visual":        800,   # SigLIP2 Text->image
        "emb_caption":   600,
        "emb_transcript":400,
        "emb_summary":   200,   # Video-level
        "bm25_ocr":      600,
        "bm25_caption":  400,
        "bm25_asr_vi":   400,
        "bm25_asr_en":   300,
        "bm25_summary":  200,
    },

    # --- Weighted RRF ---
    "rrf_k": 60.0,
    # Trọng số Khởi điểm cho profile "balanced". Các profile khác override.
    "weights_balanced": {
        "Visual": 1.00, "emb_caption": 0.65, "emb_transcript": 0.45, "bm25_ocr": 0.55,
        "bm25_caption": 0.35, "bm25_asr_vi": 0.45, "bm25_asr_en": 0.30, "Object": 0.15,
    },
    "weights_visual_heavy": {
        "Visual": 1.40, "emb_caption": 0.75, "emb_transcript": 0.20, "bm25_ocr": 0.20,
        "bm25_caption": 0.40, "bm25_asr_vi": 0.20, "bm25_asr_en": 0.15, "Object": 0.25,
    },
    "weights_ocr_heavy": {
        "Visual": 0.70, "emb_caption": 0.40, "emb_transcript": 0.30, "bm25_ocr": 1.60,
        "bm25_caption": 0.25, "bm25_asr_vi": 0.40, "bm25_asr_en": 0.20, "Object": 0.10,
    },
    "weights_speech_heavy": {
        "Visual": 0.70, "emb_caption": 0.40, "emb_transcript": 1.10, "bm25_ocr": 0.35,
        "bm25_caption": 0.25, "bm25_asr_vi": 1.20, "bm25_asr_en": 0.70, "Object": 0.10,
    },
    "weights_temporal": {   # TRAKE / Query nhiều Hành động
        "Visual": 1.20, "emb_caption": 0.70, "emb_transcript": 0.35, "bm25_ocr": 0.25,
        "bm25_caption": 0.40, "bm25_asr_vi": 0.30, "bm25_asr_en": 0.20, "Object": 0.20,
    },

    # --- Video prior: công vào score frame, Không để chế frame-level evidence ---
    "video_prior_alpha": 0.25,          # Final = frame_rrf + alpha * video_prior_norm
    "video_prior_sources": {"emb_summary": 0.6, "bm25_summary": 0.4},

    # --- Temporal NMS + diversity ---
    "nms_time_window_s": 2.0,           # trong 1 Video, 2 candidate cach < 2s -> giữ cai diem cao hơn
    # NMS chay TRUOC rerank, nen frame dung nam trong 2s cua mot frame diem cao hon
    # bi loai vinh vien truoc khi reranker/visual co co hoi cham. Tang len 2 de giu
    # 2 frame/cluster qua rerank. 1 = hanh vi hien tai. Chi doi khi da co ground
    # truth de A/B, dung doi mu.
    "nms_keep_per_cluster": 1,
    "max_per_video_top5": 2,            # Top 5 không được quá 2 frame cùng video
    "max_per_video_top20": 4,
    "max_per_video_total": 12,
    # Kích thước hồ candidate sau diversify = max_rows_per_query * fuse_pool_mult.
    # Đây mới là trần thật của rerank. 2 -> 200 (cũ), 5 -> 500.
    "fuse_pool_mult": 5,
    "neighbor_expand": 2,               # thêm +-N Keyframe lân cận cho candidate hàng đầu (biến thời gian không chắc)

    # --- Rerank (Voyage) ---
    "rerank_enabled": True,
    "rerank_model": "voyageai/rerank-2.5-lite",
    "rerank_topn": 300,                 # chỉ Rerank Top-N.
                                        # Bị chặn bởi fuse_pool_mult: rerank_stage nhận
                                        # tối đa len(fused["ranked"]) candidate.
    "rerank_weight": 0.45,              # Blend 3 nguon: rrf + visual + text-rerank
    "visual_blend_weight": 0.35,        # trong so SigLIP2 frame-level trong blend cuoi
    "rerank_doc_max_chars": 900,

    # --- MiMo Query planner (Parse / translate / Retrieval variants) ---
    "llm_model": "xiaomi/mimo-v2.5",
    "llm_max_retries": 3,
    "llm_timeout": 180,
    "translation_max_retries": 8,
    # MiMo chỉ làm Query planning: phân tích, tổ chức suy luận, dịch q_en và tạo
    # các Retrieval variants. Visual/Answer do Human Review xử lý ở NB03.
    # Mặc định không gọi VLM để tránh model tự xem Visual trước khi người review.
    "allow_vlm_verify": False,
    "vlm_topm_by_type": {"kis": 0, "qa": 0, "trake": 0},
    # Chỉ bật True trong trường hợp cần chạy chế độ hỗ trợ khẩn cấp cho query cụ thể.
    "vlm_topm_requested": 10,
    "vlm_request_file": "/kaggle/working/vlm_requests.json",
    "vlm_contact_sheet": True,          # Gui contact sheet (prev-mid-next) thấy vì 1 ảnh
    "vlm_img_max_side": 512,

    # --- Frame refinement ---
    "refine_enabled": False,
    "refine_topk": 24,                  # số candidate/Query được Refine tự động
    "refine_window_s": 3.0,             # baseline +-3s
    "refine_coarse_step_s": 0.30,
    "refine_fine_step_frames": 1,
    "refine_fine_span_frames": 8,
    "refine_max_frames_per_cand": 90,   # trần decode để không quá chậm
    "reuse_capture": True,              # tai dung VideoCapture giua cac lan read_frames_at
    "frame_cache_max": 300,             # so frame PIL giu lai (~512px RGB ~0.44MB/frame)
    "refine_rerank_gamma": 0.30,        # dung diem refine SigLIP2 de re-rank top-K

    # --- SigLIP2 Text/image encoder ---
    "siglip_model_id": "google/siglip2-giant-opt-patch16-384",
    "siglip_local_dir": None,           # trỏ tay vào Dataset offline nếu Kaggle không có Internet
    "siglip_batch": 16,

    # --- TRAKE ---
    "trake_videos": 24,                 # video sau video-rerank làm alignment; 30 nếu ưu tiên recall tối đa
    "trake_beam": 8,                    # beam nhỏ để giữ runtime ổn định
    "trake_refine_topn": 12,            # số sequence TRAKE refine trên frame thực
    "trake_seq_per_query": 100,         # số dòng CSV (sequence) tối đa xuất.
    "trake_seq_per_video": 4,           # tránh một video chiếm hết quota sequence
                                        # BTC cho 100 dòng/file và R@50+R@100 chiếm 2/5 điểm
                                        # của query -> lấy hết quota, dòng sai không bị phạt.
                                        # Lưu ý: chỉ trake_refine_topn dòng đầu được refine
                                        # frame thật; phần còn lại là frame_idx keyframe thô.
    "trake_min_gap_s": 0.20,            # event kế tiếp phải cách ít nhất (không bước cách đều)
    "trake_max_gap_s": 120.0,
    "trake_transition_w": 0.15,
    "trake_video_frame_topk": 5,        # aggregate top-k frame evidence thành video score
    "trake_video_frame_weight": 0.75,
    "trake_video_prior_weight": 0.25,
    "trake_temporal_pool": 240,         # pool trước temporal rerank; không gọi model mới

    # --- Output ---
    "max_rows_per_query": 100,
    "contact_sheet_topn": 20,           # sheets/ : giu nguyen nhu truoc (top-N)
    "candidate_image_topn": 50,         # candidates/<qid>/ : ảnh cho candidate top-N.
                                        # 100 -> ~2400 lần decode video cho cả bộ đề, tốn
                                        # 15-25 phút mà KHÔNG ăn điểm (ảnh chỉ để review).
                                        # Người review chỉ nhìn ~20 ảnh đầu mỗi query.
    "trake_sheet_seqs": 20,             # số sequence TRAKE được render ảnh

    # --- COST guard (dùng chứng ledger với NB01) ---
    "MAX_TOTAL_COST_USD": 2.00,
    "COST_WARN_RATIO": 0.80,
    "COST_HALT_RATIO": 0.90,
    "nb02_budget_usd": 0.35,            # Trần riêng cho retrieval/rerank; phần dịch bắt buộc không dùng trần này
                                        # (parse LLM ~0.0015 + rerank 200 doc ~0.0011) ->
                                        # 49 query ~ $0.13. Muc 0.08 cu bi halt quanh query 31.
    "price_embed_per_1m": 0.02,
    "price_rerank_per_1m": 0.02,
    "price_llm_in_per_1m": 0.30,        # Cập nhật theo bảng giá OpenRouter thực tế
    "price_llm_out_per_1m": 1.20,   
    "DRY_RUN": False,                    # True: không gọi API, dùng Fallback rule-based/local
}


def _extract_root():
    """/kaggle/temp Không bị lưu vào output notebook -> giải nén vào đây cho khỏi phình Output."""
    temp = Path("/kaggle/temp")
    temp.mkdir(parents=True, exist_ok=True)
    return temp

def resolve_dir(path, marker, name):
    """Trả về thư mục chứa `marker`.

    Xử lý được 3 đang mà Kaggle đưa output notebook trước sang:
      1. thư mục có sẵn marker
      2. thư mục long nhau (marker nằm ở thư mục con)
      3. Output bị đóng gói thành _output_.zip  -> tự giải nén 1 lần
    """
    import zipfile
    p = Path(path)
    if (p / marker).exists():
        return p
    dest = _extract_root() / name
    if (dest / marker).exists():
        print("[info] dùng bạn đã giải nén:", dest)
        return dest
    hit = sorted(dest.rglob(marker)) if dest.is_dir() else []
    if hit:
        print("[info] dùng bạn đã giải nén:", hit[0].parent)
        return hit[0].parent
    if p.is_dir():
        hit = sorted(p.rglob(marker))
        if hit:
            print("[info] tìm thấy", marker, "tải", hit[0].parent)
            return hit[0].parent

    def _zip_has_marker(z):
        try:
            with zipfile.ZipFile(z) as zf:
                return any(n == marker or n.endswith("/" + marker)
                           for n in zf.namelist())
        except Exception:
            return False

    if p.is_file() and p.suffix.lower() == ".zip":
        zips = [p]
    elif p.is_dir():
        zips = sorted(p.rglob("*.zip")) + sorted(p.rglob("*.ZIP"))
    else:
        zips = []

    # Nếu slug/path được Kaggle đổi, tự dò mọi ZIP đã attach.
    # Chỉ chọn ZIP thật sự chứa marker để không nhầm output của NB02/NB03.
    if not zips or not any(_zip_has_marker(z) for z in zips):
        for root in (Path("/kaggle/input"), Path("/kaggle/working")):
            if root.is_dir():
                zips.extend(sorted(root.rglob("*.zip")))
                zips.extend(sorted(root.rglob("*.ZIP")))
    seen_zips, valid_zips = set(), []
    for z in zips:
        z = Path(z)
        if str(z) not in seen_zips and z.is_file() and _zip_has_marker(z):
            seen_zips.add(str(z)); valid_zips.append(z)
    zips = valid_zips
    if zips:
        print("[info] đã dò thấy ZIP chứa", marker, "->", zips[0])
    for z in zips:
        mb = z.stat().st_size / 1e6
        print("[info] giải nén %s (%.0f MB) -> %s ... đổi vài phút" % (z.name, mb, dest))
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        hit = sorted(dest.rglob(marker))
        if hit:
            print("[info] xong ->", hit[0].parent)
            return hit[0].parent
    raise FileNotFoundError(
        "Không tìm thấy %s trong %s.\n"
        "  - Kiểm tra lại path ở panel Data bên phải.\n"
        "  - Nếu Output là File .ZIP, trỏ thẳng CFG vào file .zip đó cũng được." % (marker, path))

# Kaggle may expose A committed NB01 Output only as _output_.zip.
# Prefer A READY working directory; otherwise hand thể ZIP tổ resolve_dir below.
_art_zip_value = CFG.get("art_zip")
_art_zip = Path(_art_zip_value) if _art_zip_value else None
if not (Path(CFG["art_dir"]) / "artifact_manifest.json").exists() and _art_zip and _art_zip.is_file():
    CFG["art_dir"] = str(_art_zip)

ART = resolve_dir(CFG["art_dir"], "artifact_manifest.json", "nb01_artifacts")
# Dọn artifact/cache trung gian cũ trong /kaggle/working để không bị đóng gói lại.
if str(ART).startswith("/kaggle/temp/"):
    for _stale in (Path("/kaggle/working/nb01_artifacts"), Path("/kaggle/working/Cache")):
        if _stale.exists() and _stale.resolve() != ART.resolve():
            shutil.rmtree(_stale)
            print("[cleanup] removed stale intermediate ->", _stale)
print("[info] NB01 artifacts ->", ART)
CACHE = Path(CFG["cache_dir"])
OUT = Path(CFG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "candidates").mkdir(exist_ok=True)
(OUT / "sheets").mkdir(exist_ok=True)
(CACHE / "LLM").mkdir(parents=True, exist_ok=True)
(CACHE / "embed").mkdir(parents=True, exist_ok=True)
(CACHE / "rerank").mkdir(parents=True, exist_ok=True)

OPENROUTER_API_KEY = ""
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    try:
        from kaggle_secrets import UserSecretsClient
        OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    except Exception as e:
        print("[warn] không lay được Kaggle Secret:", type(e).__name__)
print("API key present:", bool(OPENROUTER_API_KEY), "| DRY_RUN:", CFG["DRY_RUN"])

In [ ]:
# ============================== CELL 2: HELPERS + COST LEDGER + OPENROUTER ==============================
import urllib.request, urllib.error

def jload(p):
    with open(p, "r", encoding="UTF-8") as f:
        return json.load(f)

def jdump(obj, p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="UTF-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)

def strip_accents(s):
    s = unicodedata.normalize("NFD", s or "")
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return unicodedata.normalize("NFC", s).replace("\u0111", "d").replace("\u0110", "D")

def norm_text(s):
    return unicodedata.normalize("NFC", (s or "")).strip()

_TOK = re.compile(r"[0-9a-zA-Z\u00c0-\u1ef9]+", re.UNICODE)
def tokenize(s, fold_accent=False):
    s = norm_text(s).lower()
    if fold_accent:
        s = strip_accents(s)
    return _TOK.findall(s)

def sha1(s):
    return hashlib.sha1(s.encode("UTF-8")).hexdigest()

def approx_tokens(s):
    return max(1, int(len(s) / 3.2))

def _writable(d):
    """ART có thể nằm ở /kaggle/input (Read-only) Khi dùng Output của NB01 làm Dataset."""
    try:
        p = Path(d) / ".write_test"
        p.write_text("1", encoding="UTF-8"); p.unlink()
        return True
    except Exception:
        return False

LEDGER_DIR = ART if _writable(ART) else OUT
LEDGER = LEDGER_DIR / "cost_ledger.json"
ERRLOG = LEDGER_DIR / "error_ledger.jsonl"
if LEDGER_DIR != ART:
    print("[info] ART Read-only -> ledger/error log Ghi vào", LEDGER_DIR)
    src = ART / "cost_ledger.json"
    if src.exists() and not LEDGER.exists():
        shutil.copy(src, LEDGER)      # kế thừa chỉ phí đã tiêu ở NB01

def ledger_read():
    if LEDGER.exists():
        try:
            return jload(LEDGER)
        except Exception:
            pass
    return {"total_usd": 0.0, "by_model": {}, "events": 0}

def ledger_add(model, usd, tokens=0, calls=1):
    L = ledger_read()
    L["total_usd"] = round(float(L["total_usd"]) + float(usd), 6)
    m = L["by_model"].setdefault(model, {"usd": 0.0, "tokens": 0, "calls": 0})
    m["usd"] = round(m["usd"] + float(usd), 6); m["tokens"] += int(tokens); m["calls"] += int(calls)
    L["events"] += 1
    jdump(L, LEDGER)
    return L

NB02_SPENT = {"usd": 0.0}

def budget_check(extra=0.0):
    L = ledger_read(); tot = L["total_usd"] + extra; cap = CFG["MAX_TOTAL_COST_USD"]
    if tot >= cap * CFG["COST_HALT_RATIO"]:
        raise RuntimeError("[COST HALT] %.4f >= %d%% của $%.2f" % (tot, CFG["COST_HALT_RATIO"] * 100, cap))
    if NB02_SPENT["usd"] + extra > CFG["nb02_budget_usd"]:
        raise RuntimeError("[NB02 BUDGET STOP] $%.4f > trần $%.2f" % (NB02_SPENT["usd"] + extra, CFG["nb02_budget_usd"]))
    if tot >= cap * CFG["COST_WARN_RATIO"]:
        print("[COST WARN] %.4f/%.2f USD" % (tot, cap))
    return tot

# Vuot tran -> KHONG raise nua. budget_check() raise giua vong lap khien cac query
# con lai khong co file candidate nao, du pipeline con du duong degrade (rule-based
# parser, BM25, SigLIP2 local) de chay tiep ma khong can API. api_budget_ok() bien
# viec het tien thanh "tat nhanh API, chay tiep bang tin hieu local".
API_OFF = {"on": False, "reason": ""}

def api_budget_ok(extra=0.0):
    """Soft gate: True neu con duoc goi API. Khong raise -> caller tu degrade."""
    if API_OFF["on"]:
        return False
    try:
        budget_check(extra)
        return True
    except RuntimeError as e:
        API_OFF["on"] = True
        API_OFF["reason"] = str(e)
        log_error("budget_degrade", str(e))
        print("[DEGRADE] %s\n          -> tat nhanh API, chay tiep bang tin hieu local" % e)
        return False

def log_error(kind, detail):
    with open(ERRLOG, "a", encoding="UTF-8") as f:
        f.write(json.dumps({"ts": time.time(), "NB": "02", "kind": kind,
                            "detail": str(detail)[:1500]}, ensure_ascii=False) + "\n")

def or_post(path, payload, timeout=None, retries=3):
    assert OPENROUTER_API_KEY, "Thiếu OPENROUTER_API_KEY"
    url = "https://openrouter.ai/api/v1" + path
    body = json.dumps(payload).encode("UTF-8")
    timeout = timeout or CFG["llm_timeout"]
    last = None
    for a in range(retries):
        req = urllib.request.Request(url, data=body, method="POST", headers={
            "Authorization": "Bearer " + OPENROUTER_API_KEY,
            "Content-Type": "application/json",
            "HTTP-Referer": "https://kaggle.com", "X-Title": "aic2026"})
        try:
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return json.loads(r.read().decode("UTF-8"))
        except urllib.error.HTTPError as e:
            msg = e.read().decode("UTF-8", "replace")[:500]
            last = "HTTP %d: %s" % (e.code, msg); log_error("HTTP", last)
            if e.code in (400, 401, 403, 404, 422):
                raise RuntimeError("[non-retryable] " + last)
        except Exception as e:
            last = type(e).__name__ + ": " + str(e); log_error("net", last)
        s = min(20, 2 ** a)
        print("  Retry %d/%d sau %ds (%s)" % (a + 1, retries, s, str(last)[:110]))
        time.sleep(s)
    raise RuntimeError("OpenRouter fail: %s" % last)

def minmax(x):
    x = np.asarray(x, dtype=np.float32)
    if not len(x):
        return x
    lo, hi = float(x.min()), float(x.max())
    return np.zeros_like(x) if hi - lo < 1e-9 else (x - lo) / (hi - lo)

print("ledger:", ledger_read())

In [ ]:
# ============================== CELL 3: Load ARTIFACTS ==============================
import scipy.sparse as sp
from collections import Counter, defaultdict

MAN = jload(ART / "artifact_manifest.json")
print("manifest từ NB01:", MAN["counts"])
for p in MAN.get("problems", []):
    print("  [NB01 problem]", p)

KF = pd.read_parquet(ART / "keyframes.parquet")
KF_ROW = KF.set_index("row_id")
KF_BY_VIDEO = {v: g.reset_index(drop=True) for v, g in KF.groupby("video_id")}
VIDEO_FILES = MAN.get("video_files", {})
KEYFRAME_DIRS = MAN.get("keyframe_dirs", {})
VIDEO_IDS = MAN["video_ids"]
print("Keyframe rows:", len(KF), "| videos:", len(VIDEO_IDS),
      "| Video Files:", len(VIDEO_FILES), "| kf dirs:", len(KEYFRAME_DIRS))

def _artifact_path(manifest_path):
    """Resolve NB01 manifest paths after A Notebook Output is remounted/unzipped."""
    p = Path(manifest_path)
    if p.exists():
        return p
    local = ART / p.name
    return local

DOCS = {}
for k, p in MAN["artifacts"]["docs"].items():
    pp = _artifact_path(p)
    if pp.exists():
        DOCS[k] = pd.read_parquet(pp)
        print("  docs_%-14s %7d" % (k, len(DOCS[k])))

class BM25:
    def __init__(self):
        self._Mcsc = None
    def _csc(self):
        if self._Mcsc is None:
            self._Mcsc = self.M.tocsc()
        return self._Mcsc
    def search(self, query_tokens, topk=200):
        if self.M is None or self.M.shape[0] == 0:
            return np.array([], np.int64), np.array([], np.float32)
        qc = Counter(t for t in query_tokens if t in self.vocab)
        if not qc:
            return np.array([], np.int64), np.array([], np.float32)
        scores = np.zeros(self.M.shape[0], dtype=np.float32)
        Mcsc = self._csc()
        for t in qc:
            j = self.vocab[t]
            s, e = Mcsc.indptr[j], Mcsc.indptr[j + 1]
            rows, tf = Mcsc.indices[s:e], Mcsc.data[s:e]
            scores[rows] += self.idf[j] * (tf * (self.k1 + 1)) / (tf + self.denom_doc[rows])
        k = min(topk, len(scores))
        idx = np.argpartition(-scores, k - 1)[:k]
        idx = idx[np.argsort(-scores[idx])]
        idx = idx[scores[idx] > 0]
        return idx.astype(np.int64), scores[idx]
    def score_subset(self, query_tokens, loc, n_out):
        """Điểm BM25 cho ĐÚNG một tập con doc rows, không cắt top-k toàn cục trước.

        loc: array len n_docs, loc[global_row] = chỉ số local (0..n_out-1) hoặc -1.
        video_event_scores() chỉ cần điểm của riêng 1 video; cách cũ search(topk=4000)
        rồi filter theo video làm docs của video đó chỉ sống sót 5-24% (do bảng
        mô phỏng) -> boost caption/OCR trong TRAKE bị bóp ngẫu nhiên.
        """
        out = np.zeros(int(n_out), dtype=np.float32)
        if self.M is None or self.M.shape[0] == 0 or n_out <= 0:
            return out
        qc = Counter(t for t in query_tokens if t in self.vocab)
        if not qc:
            return out
        Mcsc = self._csc()
        for t in qc:
            j = self.vocab[t]
            s, e = Mcsc.indptr[j], Mcsc.indptr[j + 1]
            rows, tf = Mcsc.indices[s:e], Mcsc.data[s:e]
            li = loc[rows]
            m = li >= 0
            if not m.any():
                continue
            r_m, tf_m, li_m = rows[m], tf[m].astype(np.float32), li[m]
            contrib = self.idf[j] * (tf_m * (self.k1 + 1)) / (tf_m + self.denom_doc[r_m])
            np.add.at(out, li_m, contrib.astype(np.float32))
        return out
    @classmethod
    def load(cls, prefix):
        o = cls(); o.M = sp.load_npz(prefix + ".npz")
        v = jload(prefix + ".vocab.json")
        # NB01 cũ có thể lưu bm25_b dưới khóa "b"; NB01 mới dùng "B".
        o.k1 = float(v.get("k1", 1.2))
        o.b = float(v.get("B", v.get("b", v.get("bm25_b", 0.75))))
        o.avgdl = float(v.get("avgdl", 1.0))
        o.vocab = v.get("vocab", {})
        st = np.load(prefix + ".stats.npz")
        o.idf, o.dl, o.denom_doc = st["idf"], st["dl"], st["denom_doc"]
        return o

# Pre-group docs theo video_id. evidence_doc() duoc goi ~rerank_topn (200) lan/query
# va truoc day scan full DataFrame cho tung bang -> bottleneck lon nhat cua cell
# rerank. Groupby mot lan o day bien no thanh dict lookup.
DOCS_BY_VIDEO = {name: {v: g for v, g in df.groupby("video_id")} for name, df in DOCS.items()}
print("docs pre-grouped theo video_id:", {k: len(v) for k, v in DOCS_BY_VIDEO.items()})

BM25_DIR = _artifact_path(MAN["artifacts"]["bm25_dir"]["path"])
BMS = {}
for f in MAN["artifacts"]["bm25_dir"]["fields"]:
    pref = str(BM25_DIR / f)
    if Path(pref + ".npz").exists():
        BMS[f] = BM25.load(pref)
print("bm25 fields:", list(BMS.keys()))

try:
    import faiss
    HAS_FAISS = True
except Exception as e:
    HAS_FAISS = False
    print("[warn] không có FAISS:", e)

VIS_INDEX, VIS_MAT = None, None
if HAS_FAISS and (ART / "faiss_siglip2.index").exists():
    VIS_INDEX = faiss.read_index(str(ART / "faiss_siglip2.index"))
    print("Visual Index ntotal:", VIS_INDEX.ntotal)
    assert VIS_INDEX.ntotal == len(KF), "Row alignment sai giữa Index và keyframes"
if (ART / "siglip2_matrix_f16.npy").exists():
    VIS_MAT = np.load(ART / "siglip2_matrix_f16.npy", mmap_mode="r")
    print("Visual matrix:", VIS_MAT.shape)
    # visual_rescore index truc tiep VIS_MAT[row_id] -> lech 1 dong la sai am tham.
    if VIS_MAT.shape[0] != len(KF):
        print("[warn] VIS_MAT lech row (%d vs %d keyframes) -> tat visual rescore"
              % (VIS_MAT.shape[0], len(KF)))
        VIS_MAT = None

TEXT_IX, TEXT_ROWMAP = {}, {}
for t in ["caption", "transcript_en", "summary"]:
    ip = ART / ("faiss_text_%s.index" % t)
    rp = ART / ("faiss_text_%s_rowmap.npy" % t)
    if HAS_FAISS and ip.exists() and rp.exists():
        TEXT_IX[t] = faiss.read_index(str(ip))
        TEXT_ROWMAP[t] = np.load(rp)
        print("  Text Index %s: ntotal=%d" % (t, TEXT_IX[t].ntotal))
if not TEXT_IX:
    print("[degrade] Chưa có Text-embedding Index -> chỉ dùng BM25 + Visual (vẫn chạy được)")

OBJ_IDX, OBJ_ENT = None, None
if (ART / "object_index.parquet").exists():
    OBJ_IDX = pd.read_parquet(ART / "object_index.parquet")
    OBJ_ENT = pd.read_parquet(ART / "object_entities.parquet")
    OBJ_BY_ENT = {e: g[["row_id", "score"]].values for e, g in OBJ_IDX.groupby("entity")}
    OBJ_ENT_SET = set(OBJ_ENT.entity)
    print("Object Index rows:", len(OBJ_IDX), "| entities:", len(OBJ_ENT))
else:
    OBJ_BY_ENT, OBJ_ENT_SET = {}, set()
    print("[degrade] Không có Object Index -> tắt Object soft boost")

## Query loading và parsing

`query_type` được lấy từ hậu tố tên file (`-kis`, `-qa`, `-trake`) và được validate lại bằng nội dung. Nếu có `E1:` thì query phải là TRAKE; nếu lệch, vẫn giữ hậu tố file nhưng ghi cảnh báo vào review package.

`q_vi` không bao giờ bị sửa. `q_en` do MiMo tạo với ràng buộc giữ nguyên tên riêng, số, OCR và thứ tự hành động. Nếu API hoặc `DRY_RUN` không hoạt động, notebook sẽ báo lỗi rõ ràng thay vì xuất `q_en = q_vi`.

In [ ]:
# ============================== CELL 4: Load QUERIES ==============================
_PRUNE_Q = re.compile(r"^(keyframes|Video|Videos_.*|Keyframes_.*|L\d+_V\d+|objects|"
                      r"__pycache__|\..*)$")

def _query_files(d):
    # Kaggle Linux phân biệt hoa/thường: dữ liệu có thể là query-*.txt hoặc Query-*.TXT.
    try:
        p = Path(d)
        return sorted(x for x in p.iterdir()
                      if x.is_file() and x.suffix.lower() == ".txt"
                      and x.stem.lower().startswith("query-"))
    except Exception:
        return []

def _count_q(d):
    return len(_query_files(d))

def _walk_dirs(p, depth):
    out = []
    if depth <= 0:
        return out
    try:
        kids = [x for x in sorted(Path(p).iterdir()) if x.is_dir()]
    except Exception:
        return out
    for k in kids:
        out.append(k)
        if not _PRUNE_Q.match(k.name):
            out.extend(_walk_dirs(k, depth - 1))
    return out

def find_query_dir():
    # 1) CFG (nếu path Đo thật sự có File Query)
    if CFG["query_dir"] and _count_q(CFG["query_dir"]):
        return CFG["query_dir"]
    # 2) manifest của NB01
    d = MAN.get("discovered", {}).get("query_dir")
    if d and _count_q(d):
        return d
    # 3) quét có giới hạn độ sâu + prune
    best, bn = None, 0
    for base in ["/kaggle/input", "."]:
        b = Path(base)
        if not b.exists():
            continue
        for r in [b] + _walk_dirs(b, int(CFG.get("query_scan_depth", 5))):
            n = _count_q(r)
            if n > bn:
                best, bn = str(r), n
    return best

QDIR = find_query_dir()
assert QDIR, "Không tìm thấy file query-*.txt trong dataset câu hỏi; kiểm tra CFG['query_dir'] hoặc attach dataset dethithunghiem."
print("Query dir:", QDIR)

QTYPE_RE = re.compile(r"-(KIS|qa|TRAKE)$", re.I)
# Dau phan cach sau so PHAI la optional: bo de thu dung "E1: ..." nhung bo de so
# tuyen that dung "E1 ..." (khong dau). Voi regex cu, de that -> events_raw = [],
# TRAKE mat duong deterministic tu file va roi ve phu thuoc hoan toan vao LLM
# (DRY_RUN / LLM loi -> trake_pipeline tra "khong co event" -> nop 0 dong).
# LLM la extractor chinh; parser dong nay chi lam fallback khi LLM/API khong dung duoc.
# Ho tro E1, Event 1, bullet list va dau gach ngang Unicode (–/—).
EVENT_RE = re.compile(r"^\s*(?:[-*•]\s*)?(?:Event|E)\s*(\d+)\s*(?:[:.\-–—)]\s*|\s+)(.+?)\s*$", re.I | re.M)

def extract_labeled_events(raw):
    events = []
    for line in (raw or '').splitlines():
        m = EVENT_RE.match(line)
        if m:
            desc = norm_text(m.group(2))
            if desc:
                events.append((int(m.group(1)), desc))
    return sorted(events, key=lambda x: x[0])

def load_queries():
    qs = []
    for p in _query_files(QDIR):
        qid = p.stem
        m = QTYPE_RE.search(qid)
        qtype = m.group(1).lower() if m else "kis"
        raw = norm_text(p.read_text(encoding="UTF-8", errors="replace"))
        events = extract_labeled_events(raw)
        warn = []
        if events and qtype != "trake":
            warn.append("nội dung có E1..EN nhưng hậu tố File là '%s'" % qtype)
        if qtype == "trake" and not events:
            warn.append("hậu tố TRAKE nhưng không Parse được E1..EN")
        if qtype == "qa" and "?" not in raw:
            warn.append("hậu tố qa nhưng không thấy đầu '?'")
        qs.append({"query_id": qid, "query_type": qtype, "q_vi": raw,
                   "events_raw": [t for _, t in sorted(events)], "warnings": warn})
    return qs

QUERIES = load_queries()
if CFG["only_queries"]:
    QUERIES = [q for q in QUERIES if q["query_id"] in CFG["only_queries"]]
print("queries:", len(QUERIES), "|",
      dict(Counter(q["query_type"] for q in QUERIES)))
for q in QUERIES:
    if q["warnings"]:
        print("  [warn]", q["query_id"], q["warnings"])
print("\nvi đủ:", QUERIES[0]["query_id"], "->", QUERIES[0]["q_vi"][:120])

In [ ]:
# ============================== CELL 5: Query PARSER (MiMo + Fallback rule-based) ==============================
STRUCT_SCHEMA_KEYS = ["query_type", "q_en", "retrieval_queries", "global_context", "visual_cues", "actions", "objects",
                      "colors", "counts", "people", "locations", "ocr_terms", "speech_terms",
                      "named_entities", "question", "answer_type", "events",
                      "hard_constraints", "soft_constraints"]

PARSE_SYS = (
    "Bạn là Query planner cho hệ thống Retrieval Video tiếng Việt. MiMo chỉ phân tích truy vấn"
    "để giúp các retriever tìm candidate tốt hơn; không xem ảnh, không tự chọn frame và không từ"
    "trả lời Visual. Trả về DUY NHẤT một JSON Object, không markdown, không giải thích.\n"
    "Bắt buộc GIỮ NGUYÊN: tên riêng, địa danh, tổ chức, chữ/số cần đọc trên biển/bảng (OCR), "
    "số lượng, màu sắc, trang phục, vật thể, quan hệ không gian, thứ tự hành động, mốc thời gian, phủ định, "
    "và nội dung câu hỏi của Q&A. Không được bỏ bớt chi tiết khi dịch sang tiếng Anh.\n"
    "IMPORTANT: q_en MUST be a complete, natural English translation of q_vi. Do not copy Vietnamese prose into q_en. Preserve only proper names and exact OCR text when needed.\n"
    "Schema:\n"
    "{\"query_type\":\"kis|qa|trake\", \"q_en\":str,"
    " \"retrieval_queries\":[str], \"global_context\":str,"
    " \"visual_cues\":[str], \"actions\":[str],"
    " \"objects\":[str], \"colors\":[str], \"counts\":[{\"what\":str,\"N\":int}],"
    " \"people\":[str], \"locations\":[str], \"ocr_terms\":[str], \"speech_terms\":[str],"
    " \"named_entities\":[str], \"question\":str, \"answer_type\":\"number|name|color|text|time|other\","
    " \"events\":[str], \"hard_constraints\":[str], \"soft_constraints\":[str]}\n"
    "For TRAKE, extract EVERY event from q_vi, including labels written as E1, Event 1, bullets, or dash-separated lines.\n"
    "Keep event order exactly as written, do not merge or split events, and never return an empty events list for a TRAKE query.\n"
    "retrieval_queries gồm 3-5 cách diễn đạt ngắn, trung thành với truy vấn: bạn dịch đầy đủ,"
    "góc nhìn Visual, góc nhìn speech/OCR và góc nhìn event/temporal nếu có. Không thêm chi tiết"
    "không có trong q_vi; không đổi tên riêng, con số, phủ định, thứ tự hoặc rang bước."
    "actions và events phải đúng thứ tự thời gian như trong truy vấn."
    "Object detection là soft constraint -> đặt vào soft_constraints."
)

def llm_cache_get(key):
    p = CACHE / "LLM" / (key[:2]) / (key + ".json")
    if p.exists():
        try:
            return jload(p)
        except Exception:
            return None
    return None

def llm_cache_put(key, val):
    p = CACHE / "LLM" / (key[:2]) / (key + ".json")
    jdump(val, p)

def llm_chat(messages, max_tokens=900, temperature=0.0, tag="LLM", json_mode=True, required=False):
    """Gọi MiMo qua OpenRouter, có Cache content-addressed. Trả về (text, usage)."""
    payload = {"model": CFG["llm_model"], "messages": messages,
               "max_tokens": max_tokens, "temperature": temperature}
    if json_mode:
        payload["response_format"] = {"type": "json_object"}
    key = sha1(CFG["llm_model"] + "|v1|" + json.dumps(payload, ensure_ascii=False, sort_keys=True))
    hit = llm_cache_get(key)
    if hit is not None:
        return hit.get("text", ""), hit.get("usage", {})
    if CFG["DRY_RUN"]:
        return None, {"dry_run": True}
    est_in = sum(approx_tokens(json.dumps(m.get("content"))[:20000]) for m in messages)
    est = est_in / 1e6 * CFG["price_llm_in_per_1m"] + max_tokens / 1e6 * CFG["price_llm_out_per_1m"]
    # Dịch q_en là bắt buộc và rất rẻ; không để budget guard của các nhánh
    # retrieval làm rơi dịch về q_vi. Các API khác vẫn dùng api_budget_ok().
    if not required and not api_budget_ok(est):
        return None, {"budget_off": True}
    res = or_post("/chat/completions", payload, retries=CFG["llm_max_retries"])
    txt = ((res.get("choices") or [{}])[0].get("message") or {}).get("content") or ""
    u = res.get("usage") or {}
    ti = int(u.get("prompt_tokens", est_in)); to = int(u.get("completion_tokens", 0))
    cost = ti / 1e6 * CFG["price_llm_in_per_1m"] + to / 1e6 * CFG["price_llm_out_per_1m"]
    NB02_SPENT["usd"] += cost
    ledger_add(CFG["llm_model"], cost, ti + to, 1)
    llm_cache_put(key, {"text": txt, "usage": {"in": ti, "out": to, "usd": cost}})
    return txt, {"in": ti, "out": to, "usd": cost}

def parse_json_loose(txt):
    """MiMo có thể bọc JSON trong markdown -> bọc tách ẩn toàn."""
    if not txt:
        return None
    t = txt.strip()
    t = re.sub(r"^```(?:json)?", "", t).strip()
    t = re.sub(r"```$", "", t).strip()
    i, j = t.find("{"), t.rfind("}")
    if i < 0 or j <= i:
        return None
    try:
        return json.loads(t[i:j + 1])
    except Exception:
        return None

_VI_COMMON_WORDS = {
    "mot", "nguoi", "dan", "ong", "ba", "tre", "em", "phu", "nu", "nam",
    "mac", "ao", "quan", "dang", "di", "dung", "ngoi", "chay", "cam", "deo",
    "noi", "nhin", "trong", "ngoai", "tren", "duoi", "ben", "gan", "trai", "phai",
    "va", "hoac", "co", "khong", "mau", "do", "xanh", "vang", "trang", "den",
    "tim", "cam", "tui", "xe", "cua", "ban", "ghe", "so", "chu", "bien", "bang",
    "thay", "bao", "nhieu", "ai", "cai", "con", "chiec", "nao", "dau", "sau",
    "truoc", "thu", "nhat", "tiep", "theo", "khi", "duoc", "liet", "ke", "tim",
    "cho", "biet", "hay", "canh", "phut", "giay", "tren", "duoi", "day", "la",
    "ban", "dich", "nhung", "cac", "voi", "tu", "nay", "kia", "mot",
}

def q_en_is_english(text, source_vi=""):
    """Reject an unchanged or mostly Vietnamese q_en before caching it."""
    t = norm_text(text)
    if not t or not re.search(r"[A-Za-z]", t):
        return False
    # Quoted OCR may intentionally remain in the original language.
    probe = re.sub(r'"[^"]*"', " ", t)
    words = re.findall(r"[a-z]+", strip_accents(probe).lower())
    hits = sum(1 for w in words if w in _VI_COMMON_WORDS)
    same = bool(source_vi) and t.casefold() == norm_text(source_vi).casefold()
    if same:
        return False
    if hits >= 2 or (len(words) >= 4 and hits / max(1, len(words)) >= 0.25):
        return False
    return True
def validate_struct(d, q):
    """Validate + coerce về schema On định. Field thiếu -> Giá trị rỗng, không crash."""
    if not isinstance(d, dict):
        d = {}
    out = {}
    out["query_type"] = q["query_type"]            # Luôn lấy từ hậu tố File (đã validate ở CELL 4)
    out["q_vi"] = q["q_vi"]                        # Không báo giờ sửa q_vi
    out["q_en"] = norm_text(d.get("q_en") or "")
    rq = d.get("retrieval_queries") or []
    if isinstance(rq, str):
        rq = [rq]
    out["retrieval_queries"] = []
    for x in rq[:6]:
        x = norm_text(str(x))
        if x and x not in out["retrieval_queries"]:
            out["retrieval_queries"].append(x)
    if out["q_en"] and out["q_en"] not in out["retrieval_queries"]:
        out["retrieval_queries"].insert(0, out["q_en"])
    out["retrieval_queries"] = out["retrieval_queries"][:6]
    out["global_context"] = norm_text(d.get("global_context") or "")
    for k in ["visual_cues", "actions", "objects", "colors", "people", "locations",
              "ocr_terms", "speech_terms", "named_entities", "events",
              "hard_constraints", "soft_constraints"]:
        v = d.get(k) or []
        if isinstance(v, str):
            v = [v]
        out[k] = [norm_text(str(x)) for x in v if norm_text(str(x))]
    cnts = []
    for c in (d.get("counts") or []):
        if isinstance(c, dict) and "N" in c:
            try:
                cnts.append({"what": norm_text(str(c.get("what", ""))), "N": int(c["N"])})
            except Exception:
                pass
    out["counts"] = cnts
    out["question"] = norm_text(d.get("question") or "")
    at = norm_text(d.get("answer_type") or "").lower()
    out["answer_type"] = at if at in ("number", "name", "color", "text", "time", "other") else "text"
    # TRAKE: events từ file luôn thắng MiMo nếu File đã có E1..EN
    if q["query_type"] == "trake":
        llm_events = list(out.get("events") or [])
        if llm_events and (not q["events_raw"] or len(llm_events) == len(q["events_raw"])):
            out["events"] = llm_events
            out["_event_source"] = "MiMo"
        elif q["events_raw"]:
            out["events"] = list(q["events_raw"])
            out["_event_source"] = "labeled_fallback"
        else:
            out["events"] = []
            out["_event_source"] = "missing"
    if q["query_type"] == "qa" and not out["question"]:
        mm = re.findall(r"([^.?!]*\?)", q["q_vi"])
        out["question"] = norm_text(mm[-1]) if mm else q["q_vi"]
    return out

_NUM_WORDS = {"mot": 1, "hai": 2, "ba": 3, "bon": 4, "nam": 5, "sau": 6, "bay": 7,
              "tam": 8, "chin": 9, "muoi": 10}
_COLORS = ["do", "xanh", "vang", "trang", "den", "cam", "tim", "hong", "nau", "xam", "bac"]

def rule_based_struct(q):
    """Fallback không dùng API. Không tốt bằng MiMo nhưng không được để query nào bị bỏ."""
    txt = q["q_vi"]; fold = strip_accents(txt).lower()
    toks = tokenize(txt)
    ocr_terms = re.findall(r"[\"\u201c\u2018\u2019\u201d]([^\"\u201c\u201d]{2,60})[\"\u201d]", txt)
    ocr_terms += re.findall(r"\B(\d{1,2}[:/]\d{2}(?:[:/]\d{2})?)\B", txt)
    caps = re.findall(r"\B([A-Z\u00c0-\u1ef9][\w\u00c0-\u1ef9]+(?:\s+[A-Z\u00c0-\u1ef9][\w\u00c0-\u1ef9]+){0,3})", txt)
    colors = [c for c in _COLORS if c in fold]
    counts = []
    for w, n in _NUM_WORDS.items():
        if re.search(r"\B" + w + r"\B", fold):
            counts.append({"what": "", "N": n})
    for m in re.findall(r"\B(\d{1,3})\B", txt):
        counts.append({"what": "", "N": int(m)})
    question = ""
    if q["query_type"] == "qa":
        mm = re.findall(r"([^.?!]*\?)", txt)
        question = norm_text(mm[-1]) if mm else txt
    return validate_struct({
        "q_en": txt,                       # không dịch được offline -> giữ q_vi
        "retrieval_queries": [txt],
        "global_context": txt[:300],
        "visual_cues": [], "actions": [], "objects": [], "colors": colors,
        "counts": counts, "people": [], "locations": [],
        "ocr_terms": [norm_text(x) for x in ocr_terms],
        "speech_terms": [], "named_entities": [norm_text(x) for x in caps][:8],
        "question": question, "answer_type": "text",
        "events": list(q["events_raw"]),
        "hard_constraints": [], "soft_constraints": ["Object detection là soft"],
    }, q)

def parse_query(q):
    return parse_query_required(q)
    """Required English translation + structured planning; never silently fall back."""
    """MiMo structured Output + validate; Retry hữu hạn; Fallback rule-based."""
    user = ("query_type từ tên File: %s\n"
            "Truy vấn tiếng Việt (q_vi), giữ nguyên mỗi chi tiết:\n<<<%s>>>\n"
            % (q["query_type"], q["q_vi"]))
    if q["events_raw"]:
        user += "Chuỗi event đã tách sẵn (giữ đúng thứ tự):\n" + \
                "\n".join("E%d: %s" % (i + 1, e) for i, e in enumerate(q["events_raw"])) + "\n"
    for attempt in range(CFG["llm_max_retries"]):
        try:
            txt, _ = llm_chat([{"role": "system", "content": PARSE_SYS},
                               {"role": "user", "content": user}],
                              max_tokens=1100, tag="Parse")
        except RuntimeError as e:
            log_error("parse_llm", "%s: %s" % (q["query_id"], e)); break
        if txt is None:
            break                              # DRY_RUN
        d = parse_json_loose(txt)
        if d:
            s = validate_struct(d, q)
            if q["query_type"] == "trake" and not s.get("events"):
                log_error("parse_schema", "%s: MiMo khÃ´ng extract Ä‘Æ°á»£c TRAKE events" % q["query_id"])
                continue
            s["_parser"] = "MiMo"
            return s
        log_error("parse_schema", "%s attempt %d" % (q["query_id"], attempt))
    s = rule_based_struct(q)
    if q["query_type"] == "trake":
        s["_event_source"] = "labeled_fallback" if s.get("events") else "missing"
    if q["query_type"] == "trake" and not s.get("events"):
        s["_parse_error"] = "TRAKE events empty after MiMo and labeled fallback"
    s["_parser"] = "rule_based" + ("_dry_run" if CFG["DRY_RUN"] else "_fallback")
    return s

def parse_query_required(q):
    """Call the LLM until q_en passes validation; never silently use q_vi."""
    user = ("query_type from filename: %s\n"
            "Vietnamese query q_vi; preserve every detail:\n<<<%s>>>\n"
            % (q["query_type"], q["q_vi"]))
    if q["events_raw"]:
        user += "Pre-split events in exact order:\n" + \
                "\n".join("E%d: %s" % (i + 1, e) for i, e in enumerate(q["events_raw"])) + "\n"
    last_error = "unknown"
    max_attempts = int(CFG.get("translation_max_retries", 8))
    for attempt in range(max_attempts):
        correction = "" if attempt == 0 else (
            "\nRETRY %d: The previous q_en was rejected. Rewrite q_en as a complete natural "
            "English translation. Do not copy Vietnamese prose. Return the full JSON again."
            % attempt)
        try:
            txt, _ = llm_chat([{"role": "system", "content": PARSE_SYS},
                               {"role": "user", "content": user + correction}],
                              max_tokens=1100, tag="ParseRequired", required=True)
        except Exception as e:
            last_error = type(e).__name__ + ": " + str(e)
            log_error("parse_llm", "%s attempt %d: %s" % (q["query_id"], attempt + 1, last_error))
            continue
        if txt is None:
            last_error = "LLM returned no content (DRY_RUN is enabled)"
            log_error("parse_llm", "%s attempt %d: %s" % (q["query_id"], attempt + 1, last_error))
            continue
        d = parse_json_loose(txt)
        if not d:
            last_error = "invalid JSON"
            log_error("parse_schema", "%s attempt %d: %s" % (q["query_id"], attempt + 1, last_error))
            continue
        s = validate_struct(d, q)
        if not q_en_is_english(s.get("q_en"), q["q_vi"]):
            last_error = "q_en is empty, unchanged, or mostly Vietnamese"
            log_error("translation_validation", "%s attempt %d: %s" % (q["query_id"], attempt + 1, last_error))
            continue
        if q["query_type"] == "trake" and not s.get("events"):
            last_error = "TRAKE events are empty"
            log_error("parse_schema", "%s attempt %d: %s" % (q["query_id"], attempt + 1, last_error))
            continue
        s["_parser"] = "MiMo"
        return s
    raise RuntimeError("[TRANSLATION REQUIRED] %s failed after %d attempts: %s" %
                       (q["query_id"], max_attempts, last_error))

PARSED = {}
pfile = OUT / "parsed_queries.json"
if pfile.exists() and not CFG["force_requery"]:
    PARSED = jload(pfile)
    print("loaded parsed_queries.json:", len(PARSED))
for q in QUERIES:
    if q["query_id"] in PARSED:
        _cached = PARSED[q["query_id"]]
        # Khi chạy live, không dùng lại kết quả rule-based/DRY_RUN cũ; phải để MiMo
        # tạo q_en + retrieval_queries thật sự.
        _trake_events_ready = (_cached.get("query_type") != "trake"
                              or bool(_cached.get("events")))
        _translation_ready = q_en_is_english(_cached.get("q_en"), q["q_vi"])
        _cache_ready = bool(_cached.get("retrieval_queries")) and _translation_ready and _trake_events_ready and (
            CFG["DRY_RUN"] or _cached.get("_parser") == "MiMo")
        if _cache_ready:
            continue
    PARSED[q["query_id"]] = parse_query(q)
    jdump(PARSED, pfile)
print("\nparsers:", dict(Counter(v.get("_parser") for v in PARSED.values())))
_q = QUERIES[0]
print("\nvi đủ Parse", _q["query_id"], "->")
print(json.dumps(PARSED[_q["query_id"]], ensure_ascii=False, indent=1)[:900])

## SigLIP2 Text encoder

Phải dùng đúng checkpoint `google/siglip2-giant-opt-patch16-384` để query text nằm cùng không gian với keyframe embedding.

Nếu không load được do thiếu Internet hoặc Dataset offline, nhánh Visual tự động tắt và pipeline chạy bằng Text branch. Notebook ghi cảnh báo vào review package và không đổi sang model khác.

In [ ]:
# ============================== CELL 6: SIGLIP2 Text/IMAGE ENCODER ==============================
SIG = {"OK": False, "model": None, "tok": None, "proc": None, "device": "cpu", "reason": ""}

def load_siglip():
    if SIG["OK"]:
        return SIG
    try:
        import torch
        from transformers import AutoModel, AutoTokenizer, AutoProcessor
        mid = CFG["siglip_local_dir"] or CFG["siglip_model_id"]
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if dev == "cuda" else torch.float32
        tok = AutoTokenizer.from_pretrained(mid)
        proc = AutoProcessor.from_pretrained(mid)
        model = AutoModel.from_pretrained(mid, torch_dtype=dtype).to(dev).eval()
        SIG.update({"OK": True, "model": model, "tok": tok, "proc": proc,
                    "device": dev, "torch": torch, "dtype": dtype})
        print("siglip2 loaded On", dev, "| dtype", dtype)
    except Exception as e:
        SIG["reason"] = type(e).__name__ + ": " + str(e)
        print("[degrade] Không Load được SigLIP2:", SIG["reason"])
        print("          -> tắt nhánh Visual; đặt CFG['siglip_local_dir'] trỏ vào Dataset offline nếu cần")
    return SIG

def _feature_tensor(value, torch):
    """Normalize Transformers 4.x tensors and Transformers 5.x Model outputs."""
    if torch.is_tensor(value):
        return value
    for name in ("pooler_output", "text_embeds", "image_embeds"):
        candidate = getattr(value, name, None)
        if candidate is not None and torch.is_tensor(candidate):
            return candidate
    hidden = getattr(value, "last_hidden_state", None)
    if hidden is not None and torch.is_tensor(hidden):
        return hidden[:, 0, :] if hidden.ndim == 3 else hidden
    if isinstance(value, (tuple, list)):
        for candidate in value:
            if torch.is_tensor(candidate):
                return candidate
    raise TypeError("SigLIP feature Output is not A tensor: " + type(value).__name__)

_TEXTVEC_MEMO = {}

def siglip_text_vec(texts):
    """Trả về (N,1536) float32 L2-normalized, hoặc None nếu không có model.

    Co memo vi refine_candidate goi ham nay MOT LAN CHO MOI CANDIDATE (refine_topk
    = 24/query) voi cung bo retrieval_queries -> khong memo thi encode lai 24 lan
    y het nhau. Query vector nho (vai KB) nen giu het trong RAM la an toan.
    """
    s = load_siglip()
    if not s["OK"]:
        return None
    memo_key = tuple(texts)
    hit = _TEXTVEC_MEMO.get(memo_key)
    if hit is not None:
        return hit
    torch = s["torch"]
    outs = []
    with torch.no_grad():
        for i in range(0, len(texts), CFG["siglip_batch"]):
            batch = texts[i:i + CFG["siglip_batch"]]
            enc = s["tok"](batch, padding="max_length", truncation=True,
                           max_length=64, return_tensors="pt").to(s["device"])
            v = _feature_tensor(s["model"].get_text_features(**enc), torch)
            v = v / v.norm(dim=-1, keepdim=True)
            outs.append(v.float().cpu().numpy())
    res = np.concatenate(outs, 0).astype(np.float32)
    _TEXTVEC_MEMO[memo_key] = res
    return res

def siglip_image_vec(pil_images):
    s = load_siglip()
    if not s["OK"] or not pil_images:
        return None
    torch = s["torch"]
    outs = []
    with torch.no_grad():
        for i in range(0, len(pil_images), CFG["siglip_batch"]):
            batch = pil_images[i:i + CFG["siglip_batch"]]
            enc = s["proc"](images=batch, return_tensors="pt").to(s["device"])
            if s["dtype"] == torch.float16:
                enc["pixel_values"] = enc["pixel_values"].half()
            v = _feature_tensor(s["model"].get_image_features(**enc), torch)
            v = v / v.norm(dim=-1, keepdim=True)
            outs.append(v.float().cpu().numpy())
    return np.concatenate(outs, 0).astype(np.float32)

# Smoke test (nhẹ): chỉ kiểm tra shape
_t = siglip_text_vec(["A man in A red shirt speaking at A press conference"])
print("SigLIP Text vec:", None if _t is None else _t.shape)
VISUAL_OK = _t is not None and VIS_INDEX is not None

In [ ]:
# ============================== CELL 7: Text-EMBEDDING Query (OpenRouter) ==============================
def embed_query_texts(texts):
    """Text-embedding-3-small cho Query. Cache theo hash. DRY_RUN -> None."""
    out = np.zeros((len(texts), 1536), dtype=np.float32)
    todo = []
    for i, t in enumerate(texts):
        k = sha1("openai/text-embedding-3-small|v1|" + t)
        p = CACHE / "embed" / k[:2] / (k + ".npy")
        if p.exists():
            try:
                out[i] = np.load(p); continue
            except Exception:
                pass
        todo.append(i)
    if todo and CFG["DRY_RUN"]:
        return None
    if todo:
        est = sum(approx_tokens(texts[i]) for i in todo) / 1e6 * CFG["price_embed_per_1m"]
        if not api_budget_ok(est):
            return None                    # nhanh text-embedding tu tat, khong crash
        try:
            res = or_post("/embeddings", {"model": "openai/text-embedding-3-small",
                                          "input": [texts[i][:8000] for i in todo]}, retries=3)
        except RuntimeError as e:
            log_error("query_embed", str(e)); return None
        data = res.get("data") or res.get("Data") or []
        ok_idx = set()
        for d, i in zip(data, todo):
            v = np.asarray(d.get("embedding", []), dtype=np.float32)
            if v.shape[0] != 1536:
                continue
            v = v / max(float(np.linalg.norm(v)), 1e-8)
            out[i] = v
            ok_idx.add(i)
            k = sha1("openai/text-embedding-3-small|v1|" + texts[i])
            p = CACHE / "embed" / k[:2] / (k + ".npy"); p.parent.mkdir(parents=True, exist_ok=True)
            np.save(p, v)
        u = res.get("usage") or {}
        tk = int(u.get("prompt_tokens", u.get("total_tokens", 0)) or 1)
        c = tk / 1e6 * CFG["price_embed_per_1m"]
        NB02_SPENT["usd"] += c; ledger_add("openai/text-embedding-3-small", c, tk, 1)
        # Guard: row nao khong parse duoc van la vector 0; neu dem di search FAISS thi
        # tra ve top-k nhieu ngau nhien, roi max() qua cac view co the lay chinh nhieu
        # do. -> BO han cac row 0, chi giu row that su hop le (caller chi max qua view
        # nen so luong row khong quan trong).
        bad = [i for i in todo if i not in ok_idx]
        if len(bad) == len(todo):
            log_error("query_embed", "khong parse duoc vector nao; keys=%s" % list(res)[:6])
            print("[warn] embed_query_texts: 0 vector hop le -> tat nhanh embedding")
            return None
        if bad:
            log_error("query_embed", "bo %d/%d view khong co vector hop le" % (len(bad), len(texts)))
            print("[warn] embed_query_texts: bo %d/%d view loi" % (len(bad), len(texts)))
            keep = [i for i in range(len(texts)) if i not in set(bad)]
            out = out[keep]
    return out

print("embed_query smoke:", None if CFG["DRY_RUN"] else "live")

## Retrieval branches và candidate evidence

Mỗi nhánh trả về `list[(row_id, rank, raw_score)]`. Nhánh video-level (`summary`) trả về `dict video_id -> score` và chỉ cộng điểm qua `video_prior_alpha`, không tạo candidate riêng.

Transcript segment được nối với keyframe gần timestamp nhất; `neighbor_expand` bổ sung keyframe lân cận để giảm lỗi ở ranh giới segment.

In [ ]:
# ============================== CELL 8: Retrieval branches ==============================
KF_INDEX = {(r.video_id, r.keyframe_n): r.row_id for r in KF.itertuples()}

def docs_to_rows(name, doc_rows, scores, expand=0):
    """Map Row của docs_<name> -> row_id Keyframe. Gộp chung bằng MAX."""
    df = DOCS[name]
    agg = {}
    for di, sc in zip(doc_rows, scores):
        r = df.iloc[int(di)]
        base_n = int(r.keyframe_n)
        vid = r.video_id
        if base_n < 0:
            continue
        for dn in range(-expand, expand + 1):
            rid = KF_INDEX.get((vid, base_n + dn))
            if rid is None:
                continue
            w = float(sc) * (1.0 if dn == 0 else 0.6 ** abs(dn))
            if w > agg.get(rid, -1e9):
                agg[rid] = w
    items = sorted(agg.items(), key=lambda x: -x[1])
    return [(rid, i, s) for i, (rid, s) in enumerate(items)]

def branch_visual(struct, topk=None):
    if not VISUAL_OK:
        return []
    topk = topk or CFG["topk"]["Visual"]
    # MiMo tạo nhiều view -> lấy MAX score để Retrieval bat được tổng thể và chi tiết.
    views = list(struct.get("retrieval_queries") or [struct["q_en"]])[:6]
    if struct.get("global_context"):
        views.append(struct["global_context"])
    vc = struct.get("visual_cues") or []
    if vc:
        views.append(", ".join(vc[:6]))
    acts = struct.get("actions") or []
    if acts:
        views.append(". ".join(acts[:4]))
    Q = siglip_text_vec(views)
    if Q is None:
        return []
    Dq, Iq = VIS_INDEX.search(Q.astype(np.float32), topk)
    agg = {}
    for row_s, row_i in zip(Dq, Iq):
        for s, i in zip(row_s, row_i):
            if i < 0:
                continue
            if float(s) > agg.get(int(i), -1e9):
                agg[int(i)] = float(s)
    items = sorted(agg.items(), key=lambda x: -x[1])[:topk]
    return [(rid, r, sc) for r, (rid, sc) in enumerate(items)]

def branch_text_emb(struct, table, topk, expand=0):
    if table not in TEXT_IX:
        return []
    qtexts = list(struct.get("retrieval_queries") or [struct["q_en"]])[:4]
    if table == "transcript_en" and struct.get("speech_terms"):
        qtexts.append(" ".join(struct["speech_terms"][:10]))
    Q = embed_query_texts(qtexts)
    if Q is None:
        return []
    Dq, Iq = TEXT_IX[table].search(Q.astype(np.float32), topk)
    rowmap = TEXT_ROWMAP[table]
    agg = {}
    for row_s, row_i in zip(Dq, Iq):
        for s, i in zip(row_s, row_i):
            if i < 0:
                continue
            di = int(rowmap[int(i)])
            if float(s) > agg.get(di, -1e9):
                agg[di] = float(s)
    if not agg:
        return []
    items = sorted(agg.items(), key=lambda x: -x[1])
    return docs_to_rows(table, [d for d, _ in items], [s for _, s in items], expand=expand)

def branch_bm25(struct, table, ana, topk, expand=0, extra_terms=None):
    key = "%s__%s" % (table, ana)
    if key not in BMS:
        return []
    # table tiếng Việt dùng q_vi; table tiếng Anh dùng các view q_en của MiMo.
    q = struct["q_en"] if table in ("caption", "transcript_en") else struct["q_vi"]
    if table in ("caption", "transcript_en"):
        q = q + " " + " ".join((struct.get("retrieval_queries") or [struct["q_en"]])[:4])
    else:
        # Nhanh tieng Viet (ocr / transcript_vi): KHONG pha retrieval_queries vao vi
        # chung la tieng Anh do MiMo dich -> token Anh chi them nhieu IDF. Chi bo sung
        # named_entities (ten rieng giu nguyen, trung tinh ngon ngu).
        q = q + " " + " ".join((struct.get("named_entities") or [])[:8])
    terms = list(extra_terms or [])
    text = q + " " + " ".join(terms)
    toks = tokenize(text, fold_accent=(ana == "fold"))
    di, sc = BMS[key].search(toks, topk=topk)
    if not len(di):
        return []
    return docs_to_rows(table, di, sc, expand=expand)

def branch_object(struct):
    """Soft boost: chỉ Trả về điểm cộng, Không dùng để loại candidate."""
    if not OBJ_BY_ENT:
        return []
    terms = set()
    for t in (struct.get("objects") or []) + (struct.get("visual_cues") or []):
        for w in tokenize(t):
            terms.add(w)
        terms.add(strip_accents(t).lower().strip())
    hits = {}
    ent_idf = dict(zip(OBJ_ENT.entity, OBJ_ENT.idf)) if OBJ_ENT is not None else {}
    for e in OBJ_ENT_SET:
        el = e.lower()
        if el in terms or any(w and w in el.split() for w in terms):
            arr = OBJ_BY_ENT.get(e)
            if arr is None:
                continue
            idf = float(ent_idf.get(e, 1.0))
            for rid, s in arr:
                v = float(s) * idf
                rid = int(rid)
                if v > hits.get(rid, -1e9):
                    hits[rid] = v
    items = sorted(hits.items(), key=lambda x: -x[1])[:2000]
    return [(rid, r, sc) for r, (rid, sc) in enumerate(items)]

def branch_video_prior(struct):
    """Video-level: emb_summary + bm25 summary/metadata -> dict video_id -> score [0,1]."""
    scores = defaultdict(float)
    if "summary" in TEXT_IX:
        summary_queries = list(struct.get("retrieval_queries") or [struct["q_en"]])[:4]
        Q = embed_query_texts(summary_queries)
        if Q is not None:
            Dq, Iq = TEXT_IX["summary"].search(Q.astype(np.float32), CFG["topk"]["emb_summary"])
            rowmap = TEXT_ROWMAP["summary"]; df = DOCS["summary"]
            vals = {}
            for row_s, row_i in zip(Dq, Iq):
                for s, i in zip(row_s, row_i):
                    if i < 0:
                        continue
                    vid = df.iloc[int(rowmap[int(i)])].video_id
                    vals[vid] = max(vals.get(vid, -1e9), float(s))
            if vals:
                ks = list(vals); nv = minmax([vals[k] for k in ks])
                for k, v in zip(ks, nv):
                    scores[k] += CFG["video_prior_sources"]["emb_summary"] * float(v)
    for ana in ("raw", "fold"):
        key = "summary__" + ana
        if key not in BMS:
            continue
        toks = tokenize(struct["q_vi"] + " " + " ".join((struct.get("retrieval_queries") or [struct["q_en"]])[:4]),
                        fold_accent=(ana == "fold"))
        di, sc = BMS[key].search(toks, topk=CFG["topk"]["bm25_summary"])
        if not len(di):
            continue
        df = DOCS["summary"]
        vals = {}
        for d, s in zip(di, sc):
            vid = df.iloc[int(d)].video_id
            vals[vid] = max(vals.get(vid, -1e9), float(s))
        ks = list(vals); nv = minmax([vals[k] for k in ks])
        for k, v in zip(ks, nv):
            scores[k] += 0.5 * CFG["video_prior_sources"]["bm25_summary"] * float(v)
    if scores:
        mx = max(scores.values()) or 1.0
        for k in scores:
            scores[k] /= mx
    return dict(scores)

def run_branches(struct, profile_weights):
    """Chạy mọi nhánh, trả về dict branch -> list[(row_id, rank, score)]."""
    tk = CFG["topk"]
    ocr_terms = (struct.get("ocr_terms") or []) + (struct.get("named_entities") or [])
    br = {}
    br["Visual"] = branch_visual(struct, tk["Visual"])
    br["emb_caption"] = branch_text_emb(struct, "caption", tk["emb_caption"], expand=0)
    br["emb_transcript"] = branch_text_emb(struct, "transcript_en", tk["emb_transcript"],
                                           expand=CFG["neighbor_expand"])
    br["bm25_ocr"] = branch_bm25(struct, "ocr", "fold", tk["bm25_ocr"], expand=0, extra_terms=ocr_terms)
    br_raw = branch_bm25(struct, "ocr", "raw", tk["bm25_ocr"], expand=0, extra_terms=ocr_terms)
    if br_raw:   # góp raw và ở cùng nhánh OCR bằng MAX-Rank
        merged = {}
        for rid, r, s in br["bm25_ocr"]:
            merged[rid] = min(merged.get(rid, 10 ** 9), r)
        for rid, r, s in br_raw:
            merged[rid] = min(merged.get(rid, 10 ** 9), r)
        items = sorted(merged.items(), key=lambda x: x[1])
        br["bm25_ocr"] = [(rid, i, 1.0 / (1 + r)) for i, (rid, r) in enumerate(items)]
    br["bm25_caption"] = branch_bm25(struct, "caption", "raw", tk["bm25_caption"])
    br["bm25_asr_vi"] = branch_bm25(struct, "transcript_vi", "fold", tk["bm25_asr_vi"],
                                    expand=CFG["neighbor_expand"],
                                    extra_terms=struct.get("speech_terms"))
    br["bm25_asr_en"] = branch_bm25(struct, "transcript_en", "raw", tk["bm25_asr_en"],
                                    expand=CFG["neighbor_expand"])
    br["Object"] = branch_object(struct)
    return {k: v for k, v in br.items() if v and profile_weights.get(k, 0) > 0}

print("branches READY")

In [ ]:
# ============================== CELL 9: FUSION (Weighted RRF) + Video PRIOR + NMS ==============================
def pick_profile(struct):
    """Chọn bộ trọng số. Chỉ 1 profile được chọn -> để đo đóng góp khi thử nghiệm."""
    if struct["query_type"] == "trake" or len(struct.get("actions") or []) >= 3:
        return "temporal", CFG["weights_temporal"]
    if struct.get("ocr_terms"):
        return "ocr_heavy", CFG["weights_ocr_heavy"]
    if struct.get("speech_terms") and not (struct.get("visual_cues") or struct.get("objects")):
        return "speech_heavy", CFG["weights_speech_heavy"]
    if (struct.get("visual_cues") or struct.get("colors")) and not struct.get("ocr_terms"):
        return "visual_heavy", CFG["weights_visual_heavy"]
    return "balanced", CFG["weights_balanced"]

def weighted_rrf(branches, weights, k=None):
    k = k or CFG["rrf_k"]
    fused, evidence = defaultdict(float), defaultdict(dict)
    for name, items in branches.items():
        w = float(weights.get(name, 0.0))
        if w <= 0:
            continue
        for rid, rank, sc in items:
            fused[rid] += w / (k + rank + 1.0)
            evidence[rid][name] = {"rank": int(rank), "score": round(float(sc), 5)}
    return fused, evidence

def apply_video_prior(fused, prior, alpha=None):
    alpha = CFG["video_prior_alpha"] if alpha is None else alpha
    if not prior:
        return fused
    rids = list(fused)
    # Rank-based normalize thay cho minmax: phan bo RRF co duoi dai, minmax bi vai
    # outlier top de bep -> phan lon candidate nam trong dai rat hep va alpha*prior
    # ap dao tin hieu frame-level (xep hang theo "video nao dung" thay vi "frame nao dung").
    _vals = np.asarray([fused[r] for r in rids], dtype=np.float32)
    _order = np.argsort(-_vals)
    _n = len(rids)
    base = np.empty(_n, dtype=np.float32)
    base[_order] = 1.0 - np.arange(_n, dtype=np.float32) / max(1, _n - 1)
    out = {}
    for r, b in zip(rids, base):
        vid = KF_ROW.at[r, "video_id"]
        out[r] = float(b) + alpha * float(prior.get(vid, 0.0))
    return out

def temporal_nms(ranked, window_s=None, keep_per_cluster=None):
    """Trong 1 Video, bộ candidate cach candidate diem cao hơn < window_s.

    keep_per_cluster=1 -> y het hanh vi cu. >1 -> giu them frame trong cung cluster
    thoi gian de reranker/visual co co hoi cham vao chung (xem CFG).
    """
    window_s = CFG["nms_time_window_s"] if window_s is None else window_s
    keep_n = int(CFG.get("nms_keep_per_cluster", 1) if keep_per_cluster is None
                 else keep_per_cluster)
    keep_n = max(1, keep_n)
    kept, per_video = [], defaultdict(list)   # per_video[vid] = list[[t_dai_dien, so_luong]]
    for rid, sc in ranked:
        vid = KF_ROW.at[rid, "video_id"]; t = float(KF_ROW.at[rid, "pts_time"])
        cl = None
        for c in per_video[vid]:
            if abs(t - c[0]) < window_s:
                cl = c
                break
        if cl is None:
            per_video[vid].append([t, 1]); kept.append((rid, sc))
        elif cl[1] < keep_n:
            cl[1] += 1; kept.append((rid, sc))
    return kept

def diversify(ranked):
    """Giới hạn số frame/video theo từng ngưỡng Top-k để tối ưu R@1/5/20/50/100."""
    out, cnt = [], defaultdict(int)
    for rid, sc in ranked:
        vid = KF_ROW.at[rid, "video_id"]
        pos = len(out)
        if pos < 5:
            cap = CFG["max_per_video_top5"]
        elif pos < 20:
            cap = CFG["max_per_video_top20"]
        else:
            cap = CFG["max_per_video_total"]
        if cnt[vid] >= cap:
            continue
        cnt[vid] += 1; out.append((rid, sc))
        if len(out) >= CFG["max_rows_per_query"] * int(CFG.get("fuse_pool_mult", 2)):
            break
    return out

def fuse_pipeline(struct):
    prof, W = pick_profile(struct)
    branches = run_branches(struct, W)
    fused_raw, evidence = weighted_rrf(branches, W)
    prior = branch_video_prior(struct)
    # TRAKE cần thấy candidate frame rộng, trước NMS/diversify và trước prior.
    # Video-rerank sẽ cộng prior đúng một lần ở cấp video.
    trake_ranked = sorted(fused_raw.items(), key=lambda x: -x[1])[:2000]
    fused = apply_video_prior(fused_raw, prior)
    ranked = sorted(fused.items(), key=lambda x: -x[1])[:2000]
    ranked = temporal_nms(ranked)
    ranked = diversify(ranked)
    return {"profile": prof, "weights": W, "ranked": ranked,
            "trake_ranked": trake_ranked, "evidence": evidence,
            "video_prior": prior, "branch_sizes": {k: len(v) for k, v in branches.items()}}

# Smoke test trên 1 Query
_s = PARSED[QUERIES[0]["query_id"]]
_r = fuse_pipeline(_s)
print("profile:", _r["profile"], "| branch sizes:", _r["branch_sizes"], "| Candidates:", len(_r["ranked"]))
for rid, sc in _r["ranked"][:5]:
    r = KF_ROW.loc[rid]
    print("   %.4f %s N=%d frame=%d T=%.1fs" % (sc, r.video_id, r.keyframe_n, r.frame_idx, r.pts_time))

## Voyage Rerank (chỉ trên Top-N)

Rerank là Text Reranker: chỉ chạm vào cặp `Query - Text document`, không chạm vector ảnh.

Mỗi candidate được dựng thành một evidence document gồm caption, OCR và transcript quanh timestamp. Điểm cuối là blend giữa RRF và rerank.

In [ ]:
# ============================== CELL 10: Voyage Rerank ==============================
def _doc_slice(name, vid):
    """Lat cat docs_<name> cua 1 video, lay tu DOCS_BY_VIDEO (khong scan full df)."""
    by = DOCS_BY_VIDEO.get(name)
    if not by:
        return None
    return by.get(vid)

def evidence_doc(rid, radius_s=6.0):
    """Ghep Text quanh candidate thành 1 document cho Reranker."""
    r = KF_ROW.loc[rid]
    vid, n, t = r.video_id, int(r.keyframe_n), float(r.pts_time)
    parts = []
    for name, tag in [("caption", "CAPTION"), ("ocr", "ocr")]:
        df = _doc_slice(name, vid)
        if df is None or not len(df):
            continue
        sub = df[df.keyframe_n.between(n - 1, n + 1)]
        if len(sub):
            parts.append("%s: %s" % (tag, " ".join(sub.text.astype(str).tolist())[:400]))
    for name, tag in [("transcript_vi", "SPEECH_VI"), ("transcript_en", "SPEECH_EN")]:
        df = _doc_slice(name, vid)
        if df is None or not len(df):
            continue
        sub = df[(df.start_time <= t + radius_s) & (df.end_time >= t - radius_s)]
        if len(sub):
            parts.append("%s: %s" % (tag, " ".join(sub.text.astype(str).tolist())[:400]))
    df = _doc_slice("summary", vid)
    if df is not None and len(df):
        sub = df[df.modality == "summary"]
        if len(sub):
            parts.append("VIDEO_SUMMARY: " + str(sub.text.iloc[0])[:300])
    return (" | ".join(parts))[: CFG["rerank_doc_max_chars"]] or "(no Text evidence)"

def voyage_rerank(query, docs):
    """Trả về list score cùng độ dài docs, hoặc None nếu không gọi được."""
    if not CFG["rerank_enabled"] or not docs:
        return None
    key = sha1(CFG["rerank_model"] + "|v1|" + query + "|" + "|".join(sha1(d) for d in docs))
    cp = CACHE / "rerank" / key[:2] / (key + ".json")
    if cp.exists():
        try:
            return jload(cp)["scores"]
        except Exception:
            pass
    if CFG["DRY_RUN"]:
        return None
    est = (approx_tokens(query) * len(docs) + sum(approx_tokens(d) for d in docs)) / 1e6 \
          * CFG["price_rerank_per_1m"]
    if not api_budget_ok(est):
        return None                        # rerank_stage van dung visual_rescore
    try:
        res = or_post("/rerank", {"model": CFG["rerank_model"], "query": query,
                                  "documents": docs, "top_n": len(docs)}, retries=3)
    except RuntimeError as e:
        log_error("rerank", str(e)); return None
    scores = [0.0] * len(docs)
    for item in (res.get("results") or res.get("Data") or []):
        i = int(item.get("index", -1))
        if 0 <= i < len(docs):
            scores[i] = float(item.get("relevance_score", item.get("score", 0.0)))
    u = res.get("usage") or {}
    tk = int(u.get("total_tokens", 0) or sum(approx_tokens(d) for d in docs))
    c = tk / 1e6 * CFG["price_rerank_per_1m"]
    NB02_SPENT["usd"] += c; ledger_add(CFG["rerank_model"], c, tk, 1)
    jdump({"scores": scores}, cp)
    return scores

def visual_rescore(struct, rids):
    """Cosine SigLIP2 giua query va embedding cua tung keyframe (max qua cac view).

    VIS_MAT da nam san trong RAM nen buoc nay gan nhu mien phi, va no la tin hieu
    doc lap voi text reranker -> can lai truong hop frame dung nhung khong co
    caption/OCR bi reranker text-only dim xuong.
    """
    if VIS_MAT is None or not VISUAL_OK or not rids:
        return None
    views = list(struct.get("retrieval_queries") or [struct["q_en"]])[:4]
    if struct.get("visual_cues"):
        views.append(", ".join(struct["visual_cues"][:6]))
    Q = siglip_text_vec(views)
    if Q is None:
        return None
    try:
        V = np.asarray(VIS_MAT[np.asarray(rids, dtype=np.int64)], dtype=np.float32)
    except Exception as e:
        log_error("visual_rescore", str(e))
        return None
    if V.shape[0] != len(rids):
        return None
    return (V @ Q.T).max(axis=1)

def rerank_stage(struct, ranked):
    n = min(CFG["rerank_topn"], len(ranked))
    if n == 0:
        return ranked, {"applied": False, "reason": "empty"}
    head = ranked[:n]
    rids = [rid for rid, _ in head]
    docs = [evidence_doc(rid) for rid in rids]
    q = " || ".join((struct.get("retrieval_queries") or [struct["q_en"]])[:3])
    q += ((" || QUESTION: " + struct["question"]) if struct.get("question") else "")
    sc = voyage_rerank(q, docs)
    vis = visual_rescore(struct, rids)
    if sc is None and vis is None:
        return ranked, {"applied": False, "reason": "dry_run_or_error"}

    # Blend 3 nguon. Thieu nguon nao thi chia lai trong so cho cac nguon con lai.
    w_rr = CFG["rerank_weight"] if sc is not None else 0.0
    w_vi = CFG["visual_blend_weight"] if vis is not None else 0.0
    w_rrf = max(0.0, 1.0 - CFG["rerank_weight"] - CFG["visual_blend_weight"])
    tot_w = w_rr + w_vi + w_rrf
    if tot_w <= 0:
        return ranked, {"applied": False, "reason": "zero_weights"}
    w_rr, w_vi, w_rrf = w_rr / tot_w, w_vi / tot_w, w_rrf / tot_w

    base = minmax([s for _, s in head])
    rr = minmax(sc) if sc is not None else np.zeros(n, dtype=np.float32)
    vv = minmax(vis) if vis is not None else np.zeros(n, dtype=np.float32)
    blended = [(rid, float(w_rrf * b + w_vi * v + w_rr * r))
               for rid, b, v, r in zip(rids, base, vv, rr)]
    blended.sort(key=lambda x: -x[1])
    tail_off = (blended[-1][1] if blended else 0.0) - 1e-3
    tail = [(rid, tail_off + 1e-6 * (len(ranked) - i)) for i, (rid, _) in enumerate(ranked[n:], start=n)]
    return blended + tail, {"applied": True, "N": n,
                            "weights": {"rrf": round(w_rrf, 3), "visual": round(w_vi, 3),
                                        "rerank": round(w_rr, 3)},
                            "visual_used": vis is not None,
                            "text_rerank_used": sc is not None}

def trake_video_rerank(struct, reranked, fused):
    """Aggregate frame rerank thành video score để chọn video cho TRAKE.

    Không gọi thêm API: dùng các frame score đã có, lấy trung bình top-k để tránh
    một frame nhiễu kéo cả video lên, rồi cộng video prior đúng một lần.
    """
    by_video = defaultdict(list)
    for rid, sc in reranked:
        try:
            vid = KF_ROW.at[int(rid), "video_id"]
            by_video[vid].append(float(sc))
        except Exception:
            continue
    prior = fused.get("video_prior") or {}
    # Cho summary prior cơ hội rescue video chưa có frame trong top rerank.
    for vid in prior:
        if vid in KF_BY_VIDEO:
            by_video.setdefault(vid, [])
    if not by_video:
        return {}, {"applied": False, "reason": "empty"}

    k = max(1, int(CFG.get("trake_video_frame_topk", 5)))
    frame_raw = {}
    for vid, vals in by_video.items():
        vals = sorted(vals, reverse=True)
        frame_raw[vid] = float(np.mean(vals[:k])) if vals else 0.0
    vids = list(by_video)
    frame_n = minmax([frame_raw[v] for v in vids])
    prior_n = minmax([float(prior.get(v, 0.0)) for v in vids])
    wf = float(CFG.get("trake_video_frame_weight", 0.75))
    wp = float(CFG.get("trake_video_prior_weight", 0.25))
    z = max(wf + wp, 1e-9); wf, wp = wf / z, wp / z
    scores = {v: float(wf * a + wp * b) for v, a, b in zip(vids, frame_n, prior_n)}
    order = sorted(vids, key=lambda v: -scores[v])
    return scores, {"applied": True, "candidate_videos": len(order),
                    "weights": {"frame_topk": round(wf, 3), "video_prior": round(wp, 3)},
                    "top_videos": [{"video_id": v, "score": round(scores[v], 6),
                                    "n_frame_evidence": len(by_video[v])}
                                   for v in order[:CFG["trake_videos"]]]}

print("Rerank READY | enabled:", CFG["rerank_enabled"], "| dry_run:", CFG["DRY_RUN"])

## Dense frame refinement

Keyframe chỉ dùng để khoanh vùng. Với TRAKE, đoạn cần dùng thường dưới 10 frame nên bước này là P0.

1. Đọc video gốc bằng OpenCV trong cửa sổ `+-refine_window_s` quanh `pts_time`.
2. Coarse: lấy mẫu theo `refine_coarse_step_s` và chọn frame tốt nhất bằng SigLIP2.
3. Fine: quét từng frame trong `+-refine_fine_span_frames` quanh frame coarse tốt nhất.

Chỉ refine `refine_topk` candidate đầu mỗi query để kiểm soát chi phí thời gian.

In [ ]:
# ============================== CELL 11: Dense frame refinement ==============================
try:
    import cv2
    HAS_CV2 = True
except Exception as e:
    HAS_CV2 = False
    print("[warn] không có cv2:", e)
try:
    from PIL import Image
    HAS_PIL = True
except Exception:
    HAS_PIL = False

def video_path(vid):
    return VIDEO_FILES.get(vid)

# read_frames_at() duoc goi ~150 lan/query (refine 24 cand x2 + sheets 20 +
# candidate images ~80). Moi lan truoc day mo lai file mp4 tren /kaggle/input
# (~100-300ms) => ~30s/query chi de mo file. Giu 1 capture dang mo va mot cache
# frame nho de khong decode lai cung mot moc thoi gian.
# CHU Y: khong doi cach suy frame_idx (CAP_PROP_POS_FRAMES sau seek) vi frame_idx
# la dap an nop bai. Toi uu "seek -> grab tuan tu" nam ngoai pham vi ban nay.
_CAP_CACHE = {"vid": None, "cap": None}
_FRAME_CACHE = {}
_FRAME_ORDER = []

def _get_capture(p, vid):
    """Tra ve (cap, shared). shared=True -> KHONG duoc release trong read_frames_at."""
    if not CFG.get("reuse_capture", True):
        return cv2.VideoCapture(p), False
    if _CAP_CACHE["vid"] == vid and _CAP_CACHE["cap"] is not None:
        return _CAP_CACHE["cap"], True
    if _CAP_CACHE["cap"] is not None:
        try:
            _CAP_CACHE["cap"].release()
        except Exception:
            pass
    cap = cv2.VideoCapture(p)
    _CAP_CACHE.update({"vid": vid, "cap": cap})
    return cap, True

def _release_capture():
    if _CAP_CACHE["cap"] is not None:
        try:
            _CAP_CACHE["cap"].release()
        except Exception:
            pass
    _CAP_CACHE.update({"vid": None, "cap": None})

def _fc_key(vid, t):
    return (vid, round(float(t), 2))

def _fc_put(vid, t, item):
    cap_n = int(CFG.get("frame_cache_max", 300))
    if cap_n <= 0:
        return
    k = _fc_key(vid, t)
    if k not in _FRAME_CACHE:
        _FRAME_ORDER.append(k)
    _FRAME_CACHE[k] = item
    while len(_FRAME_ORDER) > cap_n:
        _FRAME_CACHE.pop(_FRAME_ORDER.pop(0), None)

def read_frames_at(vid, times):
    """Trả về list[(frame_idx, pts_time, PIL.Image)] tải các mốc thời gian yêu cầu."""
    p = video_path(vid)
    if not p or not HAS_CV2 or not HAS_PIL:
        return []
    cap, shared = _get_capture(p, vid)
    if cap is None or not cap.isOpened():
        if shared:
            _release_capture()
        return []
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    nfr = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0.0
    out = []
    for t in times:
        if t < 0:
            continue
        hit = _FRAME_CACHE.get(_fc_key(vid, t))
        if hit is not None:
            out.append(hit)
            continue
        cap.set(cv2.CAP_PROP_POS_MSEC, float(t) * 1000.0)
        ok, fr = cap.read()
        if not ok:
            continue
        fi = int(round(cap.get(cv2.CAP_PROP_POS_FRAMES))) - 1
        if fi < 0:
            fi = int(round(t * fps)) if fps else -1
        if nfr and fi >= int(nfr):
            fi = int(nfr) - 1
        pts = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
        img = Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
        mx = CFG["vlm_img_max_side"]
        if max(img.size) > mx:
            img = img.copy(); img.thumbnail((mx, mx))
        item = (fi, float(pts if pts > 0 else t), img)
        _fc_put(vid, t, item)
        out.append(item)
    if not shared:
        cap.release()
    return out

def refine_candidate(struct, rid, text_for_scoring=None, window_s=None):
    """Trả về dict frame refine tốt nhất. nếu không refine được -> Trả về Keyframe gốc."""
    r = KF_ROW.loc[rid]
    vid, t0 = r.video_id, float(r.pts_time)
    base = {"video_id": vid, "frame_idx": int(r.frame_idx), "pts_time": t0,
            "refined": False, "reason": "", "score": None}
    if not CFG["refine_enabled"]:
        base["reason"] = "disabled"; return base
    if not video_path(vid):
        base["reason"] = "no_video_file"; return base
    s = load_siglip()
    if not s["OK"]:
        base["reason"] = "no_siglip"; return base
    W = CFG["refine_window_s"] if window_s is None else window_s
    step = CFG["refine_coarse_step_s"]
    times = list(np.arange(max(0.0, t0 - W), t0 + W + 1e-6, step))
    if len(times) > CFG["refine_max_frames_per_cand"]:
        times = list(np.linspace(max(0.0, t0 - W), t0 + W, CFG["refine_max_frames_per_cand"]))
    frames = read_frames_at(vid, times)
    if not frames:
        base["reason"] = "decode_failed"; return base
    # Dong bo voi retrieval: cham diem bang MAX qua cac view thay vi mot q_en don le.
    if text_for_scoring:
        views = [text_for_scoring]
    else:
        views = list(struct.get("retrieval_queries") or [struct["q_en"]])[:4]
    qv = siglip_text_vec(views)
    iv = siglip_image_vec([f[2] for f in frames])
    if qv is None or iv is None:
        base["reason"] = "encode_failed"; return base
    sc = (iv @ qv.T).max(axis=1)
    bi = int(np.argmax(sc))
    best_fi, best_t, _ = frames[bi]
    # Fine PASS: quet từng frame quanh diem Coarse tốt nhất
    fps = float(r.fps) if float(r.fps) > 0 else 25.0
    span = CFG["refine_fine_span_frames"]
    ftimes = [best_t + d / fps for d in range(-span, span + 1, CFG["refine_fine_step_frames"])]
    ftimes = [x for x in ftimes if x >= 0]
    fframes = read_frames_at(vid, ftimes)
    if fframes:
        iv2 = siglip_image_vec([f[2] for f in fframes])
        if iv2 is not None:
            sc2 = (iv2 @ qv.T).max(axis=1)
            j = int(np.argmax(sc2))
            if float(sc2[j]) >= float(sc[bi]):
                best_fi, best_t = fframes[j][0], fframes[j][1]
                sc = sc2; bi = j
    # đối chiếu với map-keyframes: frame_idx phải gần pts_time * fps
    exp = int(round(best_t * fps))
    if best_fi < 0 or abs(best_fi - exp) > max(5, 0.5 * fps):
        base["reason"] = "frame_idx_inconsistent(fi=%d exp=%d)" % (best_fi, exp)
        base["frame_idx"] = exp; base["pts_time"] = best_t; base["refined"] = True
        base["score"] = float(np.max(sc))
        return base
    return {"video_id": vid, "frame_idx": int(best_fi), "pts_time": float(best_t),
            "refined": True, "reason": "OK", "score": float(np.max(sc)),
            "keyframe_pts": t0, "delta_s": float(best_t - t0)}

print("Refine READY | cv2:", HAS_CV2, "| PIL:", HAS_PIL, "| Video Files:", len(VIDEO_FILES))

## MiMo Visual verification và Q&A

- MiMo chỉ làm query planning cho retrieval: phân tích câu hỏi, dịch `q_en`, tạo `retrieval_queries` và structured constraints.
- Human review là bước verify visual chính ở NB03; mặc định NB02 gọi 0 VLM.
- Nếu bắt buộc cần model xem ảnh, đặt `allow_vlm_verify=True` và chạy lại riêng query đó.
- Contact sheet gồm prev-mid-next, `frame_idx`, `pts_time`, OCR, ASR quanh timestamp và câu hỏi.
- Khi không chắc, giữ nhiều candidate row với answer khác nhau để người review quyết định.

In [ ]:
# ============================== CELL 12: CONTACT SHEET + MiMo Visual/QA ==============================
def make_contact_sheet(vid, center_time, fps, n_side=1, gap_s=0.7, label=True):
    """PIL image ghep prev-mid-next có Label video_id/frame_idx/pts_time."""
    if not HAS_PIL:
        return None, []
    times = [center_time + k * gap_s for k in range(-n_side, n_side + 1)]
    frames = read_frames_at(vid, [t for t in times if t >= 0])
    if not frames:
        return None, []
    from PIL import ImageDraw
    ws = [f[2].size[0] for f in frames]; hs = [f[2].size[1] for f in frames]
    H = max(hs); W = sum(ws)
    sheet = Image.new("RGB", (W, H + (22 if label else 0)), (16, 16, 16))
    x = 0; meta = []
    for fi, pts, img in frames:
        sheet.paste(img, (x, 0))
        if label:
            d = ImageDraw.Draw(sheet)
            d.text((x + 4, H + 4), "%s f=%d T=%.2fs" % (vid, fi, pts), fill=(255, 255, 0))
        meta.append({"frame_idx": int(fi), "pts_time": float(pts)})
        x += img.size[0]
    return sheet, meta

def img_to_data_uri(img, quality=78):
    buf = io.BytesIO(); img.convert("RGB").save(buf, format="JPEG", quality=quality)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode("ascii")

VERIFY_SYS = (
    "Bạn là trợ lý kiểm chứng thị giác cho hệ thống truy xuất Video. "
    "Chỉ đưa vào ảnh và Bằng chứng Text được đóng gói; không đoán. "
    "Trả về DUY NHẤT JSON: {\"match\":true|false, \"confidence\":0..1, "
    "\"answer\":str, \"evidence\":str, \"best_panel\":int}\n"
    "Quy tác Answer: Ngân GON, tối đa 100 ký tự, grounded vào ảnh/OCR/lỗi nội. "
    "Giữ nguyên ngôn ngữ của tên riêng và chữ trên biển/bảng (không dịch). "
    "Chuẩn hóa số và đơn vị. Nếu câu hỏi là đếm số -> Trả về chỉ số. "
    "Nếu không chắc chặn -> confidence thấp và answer là phương án khả dĩ nhất, không bịa. "
    "best_panel là chỉ số panel (0-based, từ trai sang phải) khớp nhất trong contact sheet."
)

def mimo_verify(struct, rid, refined=None):
    """Verify Visual + trả lời Q&A cho 1 candidate. Trả về dict (có thể là fallback khi DRY_RUN)."""
    r = KF_ROW.loc[rid]
    vid = r.video_id
    t = float(refined["pts_time"]) if refined and refined.get("refined") else float(r.pts_time)
    out = {"match": None, "confidence": 0.0, "answer": "", "evidence": "",
           "best_panel": -1, "source": "none"}
    sheet, meta = (None, [])
    if CFG["vlm_contact_sheet"]:
        sheet, meta = make_contact_sheet(vid, t, float(r.fps))
    if sheet is None and HAS_PIL:
        kp = r.kf_path
        if kp and Path(kp).exists():
            sheet = Image.open(kp); meta = [{"frame_idx": int(r.frame_idx), "pts_time": float(r.pts_time)}]
    ev = evidence_doc(rid)
    user = [{"type": "text", "text":
             "TRUY VẤN (vi): %s\nTRUY VẤN (EN): %s\nCÂU HỎI: %s\nRÀNG BUỘC CỨNG: %s\nRÀNG BUỘC MỀM: %s\n"
             "Bằng chứng Text quanh timestamp: %s\nPANELS: %s\n"
             "Hãy xác nhận ảnh có khớp truy vấn và trả lời câu hỏi (nếu có)."
             % (struct["q_vi"], struct["q_en"], struct.get("question", ""),
                "; ".join(struct.get("hard_constraints") or []),
                "; ".join(struct.get("soft_constraints") or []),
                ev, json.dumps(meta, ensure_ascii=False))}]
    if sheet is not None:
        user.append({"type": "image_url", "image_url": {"url": img_to_data_uri(sheet)}})
    else:
        out["evidence"] = "không render được ảnh -> chỉ dùng Text"
    try:
        txt, _ = llm_chat([{"role": "system", "content": VERIFY_SYS},
                           {"role": "user", "content": user}], max_tokens=400, tag="verify")
    except RuntimeError as e:
        log_error("mimo_verify", str(e)); out["source"] = "error"; return out
    if txt is None:
        out["source"] = "dry_run"
        return out
    d = parse_json_loose(txt) or {}
    out["match"] = bool(d.get("match")) if "match" in d else None
    try:
        out["confidence"] = max(0.0, min(1.0, float(d.get("confidence", 0.0))))
    except Exception:
        out["confidence"] = 0.0
    out["answer"] = norm_text(str(d.get("answer", "")))[:100]
    out["evidence"] = norm_text(str(d.get("evidence", "")))[:300]
    try:
        out["best_panel"] = int(d.get("best_panel", -1))
    except Exception:
        out["best_panel"] = -1
    out["source"] = "MiMo"
    out["panels"] = meta
    return out

print("MiMo verify READY | dry_run:", CFG["DRY_RUN"],
      "| topm mặc định theo loại:", CFG["vlm_topm_by_type"],
      "| Khi được yêu cầu:", CFG["vlm_topm_requested"])

## TRAKE - Video retrieval và temporal alignment

Tầng 1 dùng `global_context` và toàn bộ chuỗi event làm query để chọn `trake_videos` video candidate.

Tầng 2 tính điểm cho từng event trên các keyframe của mỗi video. Beam search giữ các sequence có thứ tự frame tăng dần; transition dùng khoảng cách thời gian hợp lý.

In [ ]:
# ============================== CELL 13: TRAKE alignment ==============================
def video_event_scores(vid, events):
    """(N_events, N_keyframes) score matrix cho 1 Video."""
    g = KF_BY_VIDEO.get(vid)
    if g is None or not len(g):
        return None, None
    rows = g.row_id.values.astype(np.int64)
    S = np.zeros((len(events), len(rows)), dtype=np.float32)
    # 1) SigLIP2 Text vs image embedding của chính Video
    if VIS_MAT is not None:
        E = siglip_text_vec(events)
        if E is not None:
            # Fancy-index theo dung rows: slice [rows[0]:rows[-1]+1] gia dinh row_id
            # lien ke VA g sort dung thu tu row_id -> lech thu tu la gan score event
            # cho keyframe sai mot cach am tham (shape check van pass).
            V = np.asarray(VIS_MAT[rows], dtype=np.float32)
            S += (E @ V.T).astype(np.float32)
    # 2) boost từ caption / OCR BM25 giới hạn trong Video
    for tbl, ana, w in [("caption", "raw", 0.35), ("ocr", "fold", 0.30)]:
        key = "%s__%s" % (tbl, ana)
        if key not in BMS or tbl not in DOCS:
            continue
        df = DOCS[tbl]
        # Cham diem TRUC TIEP tren docs cua video nay (score_subset) thay vi lay
        # global top-4000 roi filter -> khong con mat 76-95% evidence cua video.
        rows_doc = np.flatnonzero((df.video_id == vid).values).astype(np.int64)
        if not len(rows_doc):
            continue
        loc = np.full(len(df), -1, dtype=np.int64)
        loc[rows_doc] = np.arange(len(rows_doc), dtype=np.int64)
        kf_of_doc = df.keyframe_n.values[rows_doc]
        n_of_row = {int(nn): i for i, nn in enumerate(g.keyframe_n.values)}
        for ei, ev in enumerate(events):
            raw = BMS[key].score_subset(tokenize(ev, fold_accent=(ana == "fold")),
                                        loc, len(rows_doc))
            if not np.any(raw):
                continue
            nz = minmax(raw)
            for k_n, s in zip(kf_of_doc, nz):
                i = n_of_row.get(int(k_n))
                if i is not None:
                    S[ei, i] += w * float(s)
    for ei in range(S.shape[0]):
        S[ei] = minmax(S[ei])
    return S, g

def dp_align(S, times, beam=8, min_gap=0.2, max_gap=120.0, trans_w=0.15):
    """Tìm các chuỗi Index tăng dần tối đa hóa tổng score + transition. Trả về list (TOTAL, [idx...])."""
    Nn, Nk = S.shape
    if Nk < Nn:
        return []
    # Beam search theo từng event, mỗi state = (tổng diem, chuỗi Index)
    states = [(0.0, [])]
    for j in range(Nn):
        nxt = []
        for tot, seq in states:
            lo = (seq[-1] + 1) if seq else 0
            cand = list(range(lo, Nk))
            if not cand:
                continue
            sc = S[j, cand]
            order = np.argsort(-sc)[: max(beam * 3, 12)]
            for oi in order:
                i = cand[int(oi)]
                gap_bonus = 0.0
                if seq:
                    dt = times[i] - times[seq[-1]]
                    if dt < min_gap:
                        continue
                    if dt > max_gap:
                        gap_bonus = -0.5
                    else:
                        gap_bonus = trans_w * math.exp(-dt / max(max_gap, 1e-6))
                nxt.append((tot + float(S[j, i]) + gap_bonus, seq + [i]))
        if not nxt:
            return []
        nxt.sort(key=lambda x: -x[0])
        # để-dup theo prefix
        seen, ded = set(), []
        for tot, seq in nxt:
            k = tuple(seq)
            if k in seen:
                continue
            seen.add(k); ded.append((tot, seq))
            if len(ded) >= beam * 4:
                break
        states = ded[: beam * 4]
    states.sort(key=lambda x: -x[0])
    return states[:beam]

def trake_pipeline(struct, fused, video_scores=None):
    """Trả về list sequence: {video_id, frames:[...], score, per_event:[...]}"""
    events = struct.get("events") or []
    if not events:
        if struct.get("_parse_error"):
            return [], {"error": struct["_parse_error"],
                          "event_source": struct.get("_event_source", "unknown")}
        return [], {"error": "không có event"}
    # Video đã được rerank trước TRAKE. Fallback vẫn giữ candidate raw và cộng
    # prior một lần nếu notebook được gọi trực tiếp không qua run_query().
    if video_scores is not None:
        vscore = {v: float(s) for v, s in video_scores.items()}
    else:
        vscore = defaultdict(float)
        for rid, sc in fused.get("trake_ranked", fused["ranked"]):
            vid = KF_ROW.at[rid, "video_id"]
            vscore[vid] = max(vscore[vid], float(sc))
        for v, s in (fused.get("video_prior") or {}).items():
            vscore[v] += CFG["video_prior_alpha"] * float(s)
    vids = [v for v, _ in sorted(vscore.items(), key=lambda x: -x[1])[: CFG["trake_videos"]]]
    seqs, info = [], {"videos": vids, "n_events": len(events)}
    for vid in vids:
        S, g = video_event_scores(vid, events)
        if S is None:
            continue
        times = g.pts_time.values.astype(np.float32)
        vid_seq_n = 0
        for tot, idxs in dp_align(S, times, beam=CFG["trake_beam"],
                                  min_gap=CFG["trake_min_gap_s"],
                                  max_gap=CFG["trake_max_gap_s"],
                                  trans_w=CFG["trake_transition_w"]):
            per = []
            for j, i in enumerate(idxs):
                per.append({"event": j + 1, "row_id": int(g.row_id.iloc[i]),
                            "keyframe_n": int(g.keyframe_n.iloc[i]),
                            "frame_idx": int(g.frame_idx.iloc[i]),
                            "pts_time": float(g.pts_time.iloc[i]),
                            "event_score": float(S[j, i])})
            seqs.append({"video_id": vid, "score": float(tot) + 0.3 * float(vscore[vid]),
                         "per_event": per, "video_score": float(vscore[vid])})
            vid_seq_n += 1
            if vid_seq_n >= int(CFG.get("trake_seq_per_video", CFG["trake_beam"])):
                break
    seqs.sort(key=lambda x: -x["score"])
    # Giữ pool đủ rộng cho temporal rerank; chỉ cắt quota export sau đó.
    pool_n = max(int(CFG["trake_seq_per_query"]),
                 int(CFG.get("trake_temporal_pool", CFG["trake_seq_per_query"])))
    return seqs[:pool_n], info

def temporal_rerank_sequences(struct, seqs):
    """Rerank sequence sau alignment bằng event coverage + temporal coherence.

    Đây là scoring local trên evidence TRAKE đã có, không gọi thêm LLM/VLM/API.
    """
    if not seqs:
        return [], {"applied": False, "reason": "empty"}
    out, seen = [], set()
    max_gap = max(float(CFG.get("trake_max_gap_s", 120.0)), 1e-6)
    for s in seqs:
        per = s.get("per_event") or []
        vals = [float(x.get("event_score", 0.0)) for x in per]
        ts = [float(x.get("pts_time", 0.0)) for x in per]
        if not vals:
            continue
        gaps = [ts[i] - ts[i - 1] for i in range(1, len(ts))]
        valid_gaps = [g for g in gaps if g >= float(CFG.get("trake_min_gap_s", 0.2))]
        coherence = (float(np.mean([math.exp(-max(0.0, g) / max_gap)
                                     for g in valid_gaps])) if valid_gaps else 1.0)
        mean_ev = float(np.mean(vals))
        weak_ev = float(min(vals))
        before = float(s.get("score", 0.0))
        # Giữ score TRAKE gốc là tín hiệu chính; các bonus chỉ điều chỉnh tie/false positive.
        after = before + 0.20 * mean_ev + 0.20 * weak_ev + 0.10 * coherence
        key = (s.get("video_id"), tuple(int(x.get("frame_idx", -1)) for x in per))
        if key in seen:
            continue
        seen.add(key)
        s["score_before_temporal"] = before
        s["event_mean"] = mean_ev
        s["event_weakest"] = weak_ev
        s["temporal_coherence"] = coherence
        s["temporal_score"] = after
        s["score"] = after
        out.append(s)
    out.sort(key=lambda x: -float(x.get("score", 0.0)))
    return out, {"applied": True, "input": len(seqs), "output": len(out),
                  "method": "local_event_coverage_temporal_coherence"}

def trake_refine(struct, seqs, topn=None):
    """Refine từng event trên frame thật cho topn sequence đầu (đoạn dùng < 10 frame)."""
    topn = CFG["trake_refine_topn"] if topn is None else int(topn)
    events = struct.get("events") or []
    for si, s in enumerate(seqs[:topn]):
        for pe in s["per_event"]:
            ev_txt = events[pe["event"] - 1] if pe["event"] - 1 < len(events) else struct["q_en"]
            rr = refine_candidate(struct, pe["row_id"], text_for_scoring=ev_txt,
                                  window_s=min(CFG["refine_window_s"], 2.0))
            pe["refined_frame_idx"] = int(rr["frame_idx"])
            pe["refined_pts_time"] = float(rr["pts_time"])
            pe["refine_reason"] = rr["reason"]
        # bắt buộc thứ tự tăng dần sau refine
        fr = [pe.get("refined_frame_idx", pe["frame_idx"]) for pe in s["per_event"]]
        fixed, last = [], -1
        for i, f in enumerate(fr):
            if f <= last:
                f = last + 1
                s["per_event"][i]["refine_reason"] = (s["per_event"][i].get("refine_reason", "")
                                                      + "|forced_monotonic")
            fixed.append(f); last = f
        for pe, f in zip(s["per_event"], fixed):
            pe["refined_frame_idx"] = int(f)
        s["monotonic_ok"] = all(fixed[i] < fixed[i + 1] for i in range(len(fixed) - 1))
    return seqs

print("TRAKE READY | videos:", CFG["trake_videos"], "Beam:", CFG["trake_beam"])

## Chạy toàn bộ query và checkpoint

Mỗi query ghi checkpoint riêng `candidates/<query_id>.json`, nên Kaggle timeout không làm mất kết quả đã hoàn thành.

Rank được xếp theo điểm tổng hợp; TRAKE trả về nhiều sequence để reviewer có lựa chọn. Top 20/50/100 có thể mở rộng thêm frame lân cận khi timestamp chưa chắc chắn.

In [ ]:
# ============================== CELL 14: Run All QUERIES ==============================
def neighbor_rows(rid, k):
    r = KF_ROW.loc[rid]
    out = []
    for d in range(-k, k + 1):
        if d == 0:
            continue
        rr = KF_INDEX.get((r.video_id, int(r.keyframe_n) + d))
        if rr is not None:
            out.append(rr)
    return out

def build_rows_kis_qa(struct, fused, reranked, verifs, refines):
    """Tạo <=100 động ứng viện đã xếp hạng cho KIS/Q&A."""
    rows, seen = [], set()
    vmap = {rid: v for rid, v in verifs.items()}
    # Diem refine SigLIP2 chay tren frame that (khong phai keyframe dai dien) la tin
    # hieu chinh xac nhat trong pipeline -> dua vao xep hang thay vi vut di.
    # Bonus zero-sum trong nhom da refine, de khong day ca nhom len tren nhom chua refine.
    gamma = float(CFG.get("refine_rerank_gamma", 0.0))
    rf_bonus = {}
    if gamma > 0:
        rf_ids = [rid for rid, rr in refines.items()
                  if rr and rr.get("refined") and rr.get("score") is not None]
        if len(rf_ids) >= 2:
            rf_norm = minmax([float(refines[rid]["score"]) for rid in rf_ids])
            rf_bonus = {rid: gamma * (float(x) - 0.5) for rid, x in zip(rf_ids, rf_norm)}

    # boost theo MiMo verify
    scored = []
    for rid, sc in reranked:
        v = vmap.get(rid)
        b = float(sc) + rf_bonus.get(rid, 0.0)
        if v and v.get("source") == "MiMo":
            if v.get("match") is True:
                b += 0.5 + 0.5 * v.get("confidence", 0.0)
            elif v.get("match") is False:
                b -= 0.35
        scored.append((rid, b))
    scored.sort(key=lambda x: -x[1])

    def emit(rid, score, kind, extra=None):
        if rid in seen or len(rows) >= CFG["max_rows_per_query"]:
            return
        seen.add(rid)
        r = KF_ROW.loc[rid]
        rf = refines.get(rid)
        fi = int(rf["frame_idx"]) if rf and rf.get("refined") else int(r.frame_idx)
        pt = float(rf["pts_time"]) if rf and rf.get("refined") else float(r.pts_time)
        v = vmap.get(rid) or {}
        rec = {"rank": len(rows) + 1, "row_id": int(rid), "video_id": r.video_id,
               "keyframe_n": int(r.keyframe_n), "frame_idx": fi, "pts_time": round(pt, 3),
               "keyframe_frame_idx": int(r.frame_idx), "score": round(float(score), 6),
               "kind": kind, "refined": bool(rf and rf.get("refined")),
               "refine_reason": (rf or {}).get("reason", ""),
               "evidence": {k: v2 for k, v2 in (fused["evidence"].get(rid) or {}).items()},
               "video_prior": round(float((fused.get("video_prior") or {}).get(r.video_id, 0.0)), 4),
               "vlm": {k: v.get(k) for k in ("match", "confidence", "answer", "evidence", "source")} if v else {},
               "answer": v.get("answer", "") if v else "",
               "answer_source": v.get("source", "") if v else "",
               "kf_path": r.kf_path}
        if extra:
            rec.update(extra)
        rows.append(rec)

    for rid, sc in scored:
        emit(rid, sc, "primary")
        if len(rows) >= CFG["max_rows_per_query"]:
            break
    # nếu còn chỗ: thêm frame lân cận của Top candidate (biến thời gian chưa chắc)
    if len(rows) < CFG["max_rows_per_query"]:
        for rid, sc in scored[:20]:
            for nb in neighbor_rows(rid, CFG["neighbor_expand"]):
                emit(nb, sc * 0.9, "neighbor")
                if len(rows) >= CFG["max_rows_per_query"]:
                    break
            if len(rows) >= CFG["max_rows_per_query"]:
                break
    for i, r in enumerate(rows):
        r["rank"] = i + 1
    return rows

def run_query(q):
    struct = PARSED[q["query_id"]]
    t0 = time.time()
    fused = fuse_pipeline(struct)
    # TRAKE dùng candidate raw trước NMS/diversify; KIS/Q&A giữ flow hiện tại.
    rr_input = (fused.get("trake_ranked", fused["ranked"])
                if struct["query_type"] == "trake" else fused["ranked"])
    reranked, rinfo = rerank_stage(struct, rr_input)
    rinfo = dict(rinfo); rinfo["input_stage"] = ("trake_raw"
                                                     if struct["query_type"] == "trake"
                                                     else "post_nms_diversify")

    refines, verifs = {}, {}
    need_vlm, out_vlm_note = 0, "human_review_only"
    if struct["query_type"] != "trake":
        for rid, _ in reranked[: CFG["refine_topk"]]:
            refines[rid] = refine_candidate(struct, rid)
        # NB02 chỉ làm Query planning/Retrieval. Visual và Answer do Human Review ở NB03.
        # Chỉ bật allow_vlm_verify=True nếu cần chế độ hỗ trợ khẩn cấp cho một Query.
        if CFG.get("allow_vlm_verify", False):
            if q["query_id"] in VLM_REQUESTS:
                need_vlm = int(CFG["vlm_topm_requested"])
                out_vlm_note = "requested"
            else:
                need_vlm = int(CFG["vlm_topm_by_type"].get(struct["query_type"], 0))
                out_vlm_note = "optional"
            for rid, _ in reranked[:need_vlm]:
                try:
                    verifs[rid] = mimo_verify(struct, rid, refines.get(rid))
                except RuntimeError as e:
                    log_error("verify_stop", str(e)); break

    out = {"query_id": q["query_id"], "query_type": struct["query_type"],
           "q_vi": struct["q_vi"] if "q_vi" in struct else q["q_vi"], "q_en": struct["q_en"],
           "structured": struct, "warnings": q["warnings"],
           "vlm_policy": {"n_called": int(need_vlm), "mode": out_vlm_note,
                          "note": "Human Review là bước verify chính; VLM chỉ hỗ trợ"},
           "profile": fused["profile"], "weights": fused["weights"],
           "branch_sizes": fused["branch_sizes"], "rerank": rinfo,
           "runtime_s": None, "created_ts": time.time()}
    out["q_vi"] = q["q_vi"]

    if struct["query_type"] == "trake":
        # Tầng 1: video-rerank từ frame rerank đã tính, không gọi API thêm.
        video_scores, vinfo = trake_video_rerank(struct, reranked, fused)
        # Tầng 2: TRAKE alignment trên full keyframe của video đã chọn.
        seqs, info = trake_pipeline(struct, fused, video_scores=video_scores)
        # Tầng 3: temporal rerank local sau alignment, chỉ trên pool nhỏ.
        seqs, tinfo = temporal_rerank_sequences(struct, seqs)
        if CFG.get("refine_enabled", False):
            seqs = trake_refine(struct, seqs)
        info["video_rerank"] = vinfo
        info["temporal_rerank"] = tinfo
        out["trake_info"] = info
        out["sequences"] = [{
            "rank": i + 1, "video_id": s["video_id"], "score": round(s["score"], 6),
            "score_before_temporal": round(float(s.get("score_before_temporal", s["score"])), 6),
            "video_score": round(float(s.get("video_score", 0.0)), 6),
            "event_mean": round(float(s.get("event_mean", 0.0)), 6),
            "event_weakest": round(float(s.get("event_weakest", 0.0)), 6),
            "temporal_coherence": round(float(s.get("temporal_coherence", 0.0)), 6),
            "monotonic_ok": bool(s.get("monotonic_ok", True)),
            "frames": [int(pe.get("refined_frame_idx", pe["frame_idx"])) for pe in s["per_event"]],
            "per_event": s["per_event"],
        } for i, s in enumerate(seqs[: CFG["max_rows_per_query"]])]
        out["rows"] = []
    else:
        out["rows"] = build_rows_kis_qa(struct, fused, reranked, verifs, refines)
        out["sequences"] = []
    out["runtime_s"] = round(time.time() - t0, 2)
    out["cost_snapshot"] = {"nb02_spent": round(NB02_SPENT["usd"], 5),
                            "ledger_total": ledger_read()["total_usd"]}
    return out

# Danh sách Query mà người Review (NB03) yêu cầu gọi thêm MiMo Visual.
VLM_REQUESTS = set()
_vrf = Path(CFG["vlm_request_file"])
if _vrf.exists():
    try:
        VLM_REQUESTS = set(jload(_vrf).get("queries", []))
    except Exception as e:
        log_error("vlm_request_read", str(e))
print("VLM policy mặc định:", CFG["vlm_topm_by_type"],
      "| Query được yêu cầu thêm:", sorted(VLM_REQUESTS) or "(không có)")

RESULTS = {}
for qi, q in enumerate(QUERIES):
    fp = OUT / "candidates" / (q["query_id"] + ".json")
    if fp.exists() and not CFG["force_requery"]:
        _cached_result = jload(fp)
        _cached_struct = _cached_result.get("structured") or {}
        _trake_events_ready = (_cached_result.get("query_type") != "trake"
                                  or bool(_cached_struct.get("events")))
        _translation_ready = q_en_is_english(_cached_result.get("q_en"), q["q_vi"])
        _result_ready = bool(_cached_struct.get("retrieval_queries")) and _translation_ready and _trake_events_ready and (
            CFG["DRY_RUN"] or _cached_struct.get("_parser") == "MiMo")
        _old_vlm = int((_cached_result.get("vlm_policy") or {}).get("n_called", 0))
        if _result_ready and (CFG.get("allow_vlm_verify", False) or _old_vlm == 0):
            RESULTS[q["query_id"]] = _cached_result
            print("[%d/%d] skip (Checkpoint) %s" % (qi + 1, len(QUERIES), q["query_id"]))
            continue
    try:
        res = run_query(q)
    except RuntimeError as e:
        print("[%d/%d] STOP tải %s: %s" % (qi + 1, len(QUERIES), q["query_id"], e))
        log_error("run_query", "%s: %s" % (q["query_id"], e))
        break
    jdump(res, fp)
    RESULTS[q["query_id"]] = res
    nrow = len(res["rows"]) or len(res["sequences"])
    print("[%d/%d] %-24s type=%-5s profile=%-13s rows=%3d %.1fs spent=$%.4f"
          % (qi + 1, len(QUERIES), q["query_id"], res["query_type"], res["profile"],
             nrow, res["runtime_s"], NB02_SPENT["usd"]))

print("\ndone queries:", len(RESULTS), "/", len(QUERIES))
print("NB02 spent: $%.4f | ledger TOTAL: $%.4f / $%.2f"
      % (NB02_SPENT["usd"], ledger_read()["total_usd"], CFG["MAX_TOTAL_COST_USD"]))

In [ ]:
# ============================== CELL 15: PROXY METRICS / DIAGNOSTICS ==============================
# Chưa có ground truth -> dùng proxy Metric để phát hiện Pipeline hong, không phải để tuyển bộ diem.
diag = []
for qid, r in RESULTS.items():
    if r["query_type"] == "trake":
        seqs = r["sequences"]
        diag.append({"query_id": qid, "type": "trake", "n_out": len(seqs),
                     "n_videos": len(set(s["video_id"] for s in seqs)),
                     "seq_valid": sum(1 for s in seqs if s["monotonic_ok"]
                                      and len(s["frames"]) == len(r["structured"]["events"])),
                     "top1_video": seqs[0]["video_id"] if seqs else "",
                     "top1_score": seqs[0]["score"] if seqs else 0.0,
                     "runtime_s": r["runtime_s"]})
    else:
        rows = r["rows"]
        diag.append({"query_id": qid, "type": r["query_type"], "n_out": len(rows),
                     "n_videos": len(set(x["video_id"] for x in rows)),
                     "n_refined": sum(1 for x in rows if x["refined"]),
                     "n_vlm": sum(1 for x in rows if (x.get("vlm") or {}).get("source") == "MiMo"),
                     "vlm_match": sum(1 for x in rows if (x.get("vlm") or {}).get("match") is True),
                     "has_answer": sum(1 for x in rows if x.get("answer")),
                     "top1_video": rows[0]["video_id"] if rows else "",
                     "top1_score": rows[0]["score"] if rows else 0.0,
                     "video_top5": len(set(x["video_id"] for x in rows[:5])),
                     "runtime_s": r["runtime_s"]})
DIAG = pd.DataFrame(diag)
DIAG.to_csv(OUT / "diagnostics.csv", index=False)
print(DIAG.to_string(index=False))
print("\n--- cảnh báo cần người Review ưu tiên ---")
if len(DIAG):
    low = DIAG[(DIAG.get("n_vlm", 0) == 0) | (DIAG.n_out < CFG["max_rows_per_query"] * 0.5)]
    for _, r in low.iterrows():
        print("  ", r.query_id, "- ít candidate hoặc chưa verify Visual")
    print("  Toàn bộ TRAKE (%d Query) cần Review thủ công."
          % int((DIAG.type == "trake").sum()))

In [ ]:
# ============================== CELL 16: CONTACT SHEETS + Review PACKAGE ==============================
SHEET_INDEX = []

def _safe_sheet_component(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))

def _source_kf_rel(kp):
    norm = str(kp or "").replace("\\", "/").strip("/")
    parts = [x for x in norm.split("/") if x]
    return "/".join(parts[-2:]) if len(parts) >= 2 else (parts[-1] if parts else "")

def _source_image_name(kp):
    """Tên tệp ảnh keyframe gốc, được giữ rõ ràng để tiện rà soát."""
    norm = str(kp or "").replace("\\", "/").strip("/")
    return norm.split("/")[-1] if norm else ""

def _source_kf_label(kp):
    """Nhãn ngắn, có thể tìm kiếm, lấy từ đường dẫn keyframe gốc."""
    rel = _source_kf_rel(kp)
    return _safe_sheet_component(rel.replace("/", "_") or "no_source")

def save_sheet(qid, tag, vid, center_t, fps, frame_idx=None, source_kf_path=None,
               subdir="sheets", record=True):
    sheet, meta = make_contact_sheet(vid, center_t, fps, n_side=1, gap_s=0.7)
    if sheet is None:
        return None, None, []
    d = OUT / subdir / qid
    d.mkdir(parents=True, exist_ok=True)
    fi = int(frame_idx) if frame_idx is not None else int(round(float(center_t) * float(fps)))
    ts_ms = int(round(float(center_t) * 1000.0))
    src_rel = _source_kf_rel(source_kf_path)
    src_name = _source_image_name(source_kf_path)
    src_label = _source_kf_label(source_kf_path)
    fname = "%s__%s__f%d__src_%s__t%07dms.jpg" % (
        _safe_sheet_component(tag), _safe_sheet_component(vid), fi, src_label, ts_ms)
    p = d / fname
    sheet.convert("RGB").save(p, quality=80)
    rel = str(p.relative_to(OUT)).replace("\\", "/")
    if record:
        SHEET_INDEX.append({"query_id": qid, "tag": tag, "video_id": str(vid),
                            "frame_idx": fi, "pts_time": round(float(center_t), 3),
                            "source_kf_path": str(source_kf_path or ""),
                            "source_image_rel": src_rel, "source_image": src_label,
                            "source_image_name": src_name,
                            "image": rel, "image_name": p.name})
    return str(p), rel, meta

sheet_count = 0
for qid, r in RESULTS.items():
    if r["query_type"] == "trake":
        for s in r["sequences"][: CFG.get("trake_sheet_seqs", 20)]:
            for pe in s["per_event"]:
                t = pe.get("refined_pts_time", pe["pts_time"])
                fps = float(KF_ROW.at[pe["row_id"], "fps"])
                src_kf = str(KF_ROW.at[pe["row_id"], "kf_path"])
                p, prel, meta = save_sheet(qid, "S%02d_E%02d" % (s["rank"], pe["event"]),
                                     s["video_id"], t, fps,
                                     int(pe.get("refined_frame_idx", pe["frame_idx"])),
                                     src_kf)
                if p:
                    pe["sheet"] = p; pe["sheet_rel"] = prel; pe["sheet_panels"] = meta; pe["source_kf_path"] = src_kf; pe["source_image_rel"] = _source_kf_rel(src_kf); pe["source_image_name"] = _source_image_name(src_kf); pe["image_name"] = Path(prel).name if prel else Path(p).name; sheet_count += 1
    else:
        for x in r["rows"][: CFG["contact_sheet_topn"]]:
            fps = float(KF_ROW.at[x["row_id"], "fps"])
            src_kf = str(KF_ROW.at[x["row_id"], "kf_path"])
            p, prel, meta = save_sheet(qid, "R%03d" % x["rank"], x["video_id"],
                                 x["pts_time"], fps, int(x["frame_idx"]), src_kf)
            if p:
                x["sheet"] = p; x["sheet_rel"] = prel; x["sheet_panels"] = meta; x["source_kf_path"] = src_kf; x["source_image_rel"] = _source_kf_rel(src_kf); x["source_image_name"] = _source_image_name(src_kf); x["image_name"] = Path(prel).name if prel else Path(p).name; sheet_count += 1
    jdump(r, OUT / "candidates" / (qid + ".json"))
jdump(SHEET_INDEX, OUT / "sheets_manifest.json")
print("sheets manifest ->", OUT / "sheets_manifest.json")
print("contact sheets:", sheet_count, "->", OUT / "sheets")

# ---------- Ảnh cho MỌI ứng viên -> candidates/<qid>/ ----------
# sheets/ giữ nguyên ý nghĩa cũ (chỉ top-N, đều là ứng viên đã qua refine_topk).
# candidates/<qid>/ là bản ĐẦY ĐỦ: gồm cả những ứng viên nằm ngoài refine_topk
# nên chưa được refine. Ảnh nào đã tạo ở sheets/ thì SAO CHÉP sang, không giải mã lại.
cand_img, cand_new = 0, 0
for qid, r in RESULTS.items():
    cdir = OUT / "candidates" / qid
    cdir.mkdir(parents=True, exist_ok=True)
    items = []
    if r["query_type"] == "trake":
        for s in r["sequences"][: CFG.get("trake_sheet_seqs", 20)]:
            for pe in s["per_event"]:
                items.append(("S%02d_E%02d" % (s["rank"], pe["event"]), s["video_id"],
                              pe["row_id"], float(pe.get("refined_pts_time", pe["pts_time"])),
                              int(pe.get("refined_frame_idx", pe["frame_idx"])), pe))
    else:
        for x in r["rows"][: CFG.get("candidate_image_topn", 100)]:
            items.append(("R%03d" % x["rank"], x["video_id"], x["row_id"],
                          float(x["pts_time"]), int(x["frame_idx"]), x))
    for tag, vid, rid, t, fi, obj in items:
        prev = obj.get("sheet")
        if prev and Path(prev).exists():
            dstp = cdir / Path(prev).name
            if not dstp.exists():
                shutil.copy2(prev, dstp)
            obj["image"] = str(dstp.relative_to(OUT)).replace("\\", "/")
            obj["image_name"] = dstp.name
            obj.setdefault("source_image_name", _source_image_name(obj.get("source_kf_path", "")))
            cand_img += 1
            continue
        fps = float(KF_ROW.at[rid, "fps"])
        src_kf = str(KF_ROW.at[rid, "kf_path"])
        _ap, _rel, _meta = save_sheet(qid, tag, vid, t, fps, fi, src_kf,
                                      subdir="candidates", record=False)
        if _rel:
            obj["image"] = _rel
            obj["image_name"] = Path(_rel).name
            obj.setdefault("source_image_name", _source_image_name(src_kf))
            obj.setdefault("sheet_panels", _meta)
            cand_img += 1; cand_new += 1
    jdump(r, OUT / "candidates" / (qid + ".json"))
_release_capture()
_FRAME_CACHE.clear(); _FRAME_ORDER.clear()
print("Ảnh ứng viên:", cand_img, "(tạo mới:", cand_new, ") ->", OUT / "candidates")

# ---------- Trang HTML để rà soát bằng mắt, mở trực tiếp từ thư mục đã tải về ----------
def _esc(s):
    return (str(s).replace("&", "&amp;").replace("<", "&lt;")
            .replace(">", "&gt;").replace('"', "&quot;").replace("'", "&#39;"))

_CSS = """<style>
body{background:#111;color:#ddd;font:13px/1.45 "Segoe UI","Noto Sans",Arial,sans-serif;margin:0;padding:16px}
a{color:#6cf} h1{font-size:17px;margin:0 0 4px} h2{font-size:14px;color:#9ab;margin:18px 0 6px}
.q{background:#1b1b1b;border:1px solid #333;border-radius:6px;padding:10px 12px;margin-bottom:14px}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));gap:10px}
.card{background:#1b1b1b;border:1px solid #333;border-radius:6px;overflow:hidden}
.card img{width:100%;display:block;background:#000}
.meta{padding:6px 8px;font-size:11px;color:#bbb}
.img-name{color:#fff;word-break:break-all;margin-top:4px}
.source-name{color:#9ab;word-break:break-all}
.thumb{width:110px;max-height:80px;object-fit:cover;display:block}
.scroll{overflow:auto;max-height:75vh}
.nowrap{white-space:nowrap}
.rk{display:inline-block;background:#2a4;color:#000;font-weight:700;border-radius:3px;padding:0 5px;margin-right:5px}
.rk.lo{background:#666;color:#ddd} .warn{color:#fa4}
.note{color:#aaa;margin:6px 0 10px}.top-video{background:#263b2b}
table{border-collapse:collapse;font-size:12px} td,th{border:1px solid #333;padding:3px 7px;text-align:left;vertical-align:top}
</style>"""

def _posix_path(p):
    return str(p or "").replace("\\", "/")

_QUERY_TYPE_LABELS = {
    "kis": "KIS - tìm kiếm khoảnh khắc",
    "qa": "Q&A - hỏi đáp",
    "trake": "TRAKE - chuỗi sự kiện",
}

def _query_type_label(qt):
    return _QUERY_TYPE_LABELS.get(str(qt or "").lower(), str(qt or "").upper())

def _query_video_stats_for(r):
    """Thống kê video trong một truy vấn, đọc từ ứng viên/chuỗi thay vì chỉ đếm ảnh."""
    grouped = defaultdict(lambda: {"occurrences": 0, "events": 0,
                                    "ranks": [], "best_rank": None,
                                    "best_score": None})
    qt = str(r.get("query_type", "kis")).lower()
    if qt == "trake":
        source = r.get("sequences") or []
        for i, seq in enumerate(source, 1):
            vid = str(seq.get("video_id", ""))
            rank = int(seq.get("rank", i))
            event_count = len(seq.get("per_event") or seq.get("frames") or [])
            score = seq.get("score")
            st = grouped[vid]
            st["occurrences"] += 1       # 1 chuỗi = 1 lần xuất hiện
            st["events"] += event_count
            st["ranks"].append(rank)
            st["best_rank"] = rank if st["best_rank"] is None else min(st["best_rank"], rank)
            if score is not None:
                score = float(score)
                st["best_score"] = score if st["best_score"] is None else max(st["best_score"], score)
    else:
        source = r.get("rows") or []
        for i, row in enumerate(source, 1):
            vid = str(row.get("video_id", ""))
            rank = int(row.get("rank", i))
            score = row.get("score")
            st = grouped[vid]
            st["occurrences"] += 1       # 1 dòng ứng viên = 1 lần xuất hiện
            st["ranks"].append(rank)
            st["best_rank"] = rank if st["best_rank"] is None else min(st["best_rank"], rank)
            if score is not None:
                score = float(score)
                st["best_score"] = score if st["best_score"] is None else max(st["best_score"], score)
    max_occurrences = max((x["occurrences"] for x in grouped.values()), default=0)
    result = []
    for vid, st in sorted(grouped.items(), key=lambda z: (-z[1]["occurrences"], z[1]["best_rank"] or 10**9, z[0])):
        result.append({"video_id": vid, **st, "is_top": bool(st["occurrences"] == max_occurrences and max_occurrences > 0)})
    return result

_QUERY_VIDEO_STATS = {qid: _query_video_stats_for(r) for qid, r in RESULTS.items()}

def _image_names(obj, img_rel):
    img_name = obj.get("image_name") or Path(_posix_path(img_rel)).name
    src_name = obj.get("source_image_name") or _source_image_name(obj.get("source_kf_path", ""))
    return str(img_name), str(src_name)

def _card(img_rel, rank, sub, image_name="", source_name="", warn=""):
    cls = "rk" if rank <= 20 else "rk lo"
    names = '<div class="img-name"><b>tệp ảnh:</b> %s</div>' % _esc(image_name or Path(_posix_path(img_rel)).name)
    if source_name:
        names += '<div class="source-name"><b>ảnh nguồn:</b> %s</div>' % _esc(source_name)
    return ('<div class="card"><a href="%s" target="_blank"><img loading="lazy" alt="%s" src="%s"></a>'
            '<div class="meta"><span class="%s">#%d</span>%s%s%s</div></div>'
            % (_esc(img_rel), _esc(image_name), _esc(img_rel), cls, rank, _esc(sub), names,
               ('<div class="warn">%s</div>' % _esc(warn)) if warn else ""))

_index_rows = []
_review_images = []
for qid, r in RESULTS.items():
    qdir = OUT / "candidates" / qid
    qdir.mkdir(parents=True, exist_ok=True)
    cards, n_img = [], 0
    q_image_counts = Counter()
    if r["query_type"] == "trake":
        for s in r["sequences"][: CFG.get("trake_sheet_seqs", 20)]:
            for pe in s["per_event"]:
                sh = pe.get("image") or pe.get("sheet_rel")
                if not sh:
                    continue
                image_name, source_name = _image_names(pe, sh)
                img_rel = Path(_posix_path(sh)).name
                cards.append(_card(img_rel, s["rank"],
                                   "Chuỗi %02d | sự kiện %d | video=%s | khung hình=%d | thời gian=%.2f giây"
                                   % (s["rank"], pe["event"], s["video_id"],
                                      int(pe.get("refined_frame_idx", pe["frame_idx"])),
                                      float(pe.get("refined_pts_time", pe["pts_time"]))),
                                   image_name=image_name, source_name=source_name))
                _review_images.append({"query_id": qid, "query_type": r["query_type"],
                                       "video_id": str(s["video_id"]), "rank": int(s["rank"]),
                                       "event": int(pe["event"]), "image": _posix_path(sh),
                                       "image_name": image_name, "source_image_name": source_name,
                                       "frame_idx": int(pe.get("refined_frame_idx", pe["frame_idx"])),
                                       "pts_time": float(pe.get("refined_pts_time", pe["pts_time"])),
                                       "score": float(pe.get("event_score", s.get("score", 0.0)))})
                q_image_counts[str(s["video_id"])] += 1
                n_img += 1
    else:
        for x in r["rows"]:
            sh = x.get("image") or x.get("sheet_rel")
            if not sh:
                continue
            image_name, source_name = _image_names(x, sh)
            img_rel = Path(_posix_path(sh)).name
            cards.append(_card(img_rel, int(x["rank"]),
                               "video=%s | khung hình=%d | thời gian=%.2f giây | điểm=%.3f%s"
                               % (x["video_id"], int(x["frame_idx"]), float(x["pts_time"]),
                                  float(x["score"]), "" if x.get("refined") else " (thô)"),
                               image_name=image_name, source_name=source_name))
            _review_images.append({"query_id": qid, "query_type": r["query_type"],
                                   "video_id": str(x["video_id"]), "rank": int(x["rank"]),
                                   "event": "", "image": _posix_path(sh),
                                   "image_name": image_name, "source_image_name": source_name,
                                   "frame_idx": int(x["frame_idx"]), "pts_time": float(x["pts_time"]),
                                   "score": float(x["score"]), "kind": x.get("kind", "")})
            q_image_counts[str(x["video_id"])] += 1
            n_img += 1
    _qv_rows = _QUERY_VIDEO_STATS.get(qid, [])
    q_video_rows = "".join(
        "<tr><td>%s%s</td><td>%d</td><td>%s</td><td>%s</td><td>%s</td><td>%d</td></tr>"
        % (("⭐ " if st["is_top"] else ""), _esc(st["video_id"]),
           st["occurrences"],
           "; ".join("#%d" % x for x in st["ranks"]),
           "#%s" % st["best_rank"] if st["best_rank"] is not None else "-",
           "%.3f" % st["best_score"] if st["best_score"] is not None else "-",
           q_image_counts.get(st["video_id"], 0))
        for st in _qv_rows) or "<tr><td colspan='6'>Truy vấn này chưa có video ứng viên.</td></tr>"
    q_video_note = ("Với TRAKE, mỗi chuỗi sự kiện được tính là một lần xuất hiện; cột sự kiện là tổng số mốc thời gian."
                    if str(r["query_type"]).lower() == "trake" else
                    "Mỗi dòng ứng viên được tính là một lần xuất hiện của video trong truy vấn.")
    q_video_html = ("<h2>Video trong truy vấn này</h2><p class='note'>%s<br>⭐ là video xuất hiện nhiều nhất trong truy vấn.</p>"
                    "<div class='scroll'><table><tr><th>mã video</th><th>số lần xuất hiện</th><th>các hạng</th><th>hạng tốt nhất</th><th>điểm tốt nhất</th><th>ảnh rà soát</th></tr>%s</table></div>"
                    % (_esc(q_video_note), q_video_rows))
    html = ("<!doctype html><html lang='vi'><head><meta charset='UTF-8'><meta name='viewport' content='width=device-width, initial-scale=1'><title>%s</title></head><body>%s"
            "<h1>%s <small style='color:#888'>[%s] %d ảnh</small></h1>"
            "<div class='q'><b>Truy vấn tiếng Việt:</b> %s<br><b>Bản dịch tiếng Anh:</b> %s<br>"
            "<small style='color:#888'>hồ sơ=%s | %s</small></div>"
            "<p><a href='../review.html'>&larr; Tất cả truy vấn</a></p>%s<div class='grid'>%s</div></body></html>"
            % (_esc(qid), _CSS, _esc(qid), _esc(_query_type_label(r["query_type"])), n_img,
               _esc(r["q_vi"]), _esc(r.get("q_en", "")), _esc(r.get("profile", "")),
               _esc("; ".join(r.get("warnings") or []) or "không có cảnh báo"),
               q_video_html, "".join(cards)))
    (qdir / "index.html").write_text(html, encoding="utf-8")
    _index_rows.append((qid, r["query_type"], n_img,
                        len(r["rows"]) or len(r["sequences"]), r["q_vi"][:90]))

_review_image_counts = Counter((str(x["query_id"]), str(x["video_id"])) for x in _review_images)

# Tóm tắt theo truy vấn: có bao nhiêu video và video nào xuất hiện nhiều nhất.
_query_overview_rows = []
for qid, qtype, n_img, total, q_vi in _index_rows:
    stats = _QUERY_VIDEO_STATS.get(qid, [])
    leaders = [st for st in stats if st["is_top"]]
    leader_text = (", ".join("%s (%d lần)" % (_esc(st["video_id"]), st["occurrences"]) for st in leaders)
                   or "Không có video ứng viên")
    _query_overview_rows.append(
        "<tr><td><a href='candidates/%s/index.html'>%s</a></td><td>%s</td>"
        "<td>%d</td><td>%d</td><td>%d</td><td>%s</td><td>%s</td></tr>"
        % (_esc(qid), _esc(qid), _esc(_query_type_label(qtype)), n_img, total, len(stats),
           leader_text, _esc(q_vi)))
_query_overview_rows_html = "".join(_query_overview_rows)

# Chi tiết mọi video trong từng truy vấn; các dòng được sắp xếp theo tần suất giảm dần.
_query_video_rows_html = "".join(
    "<tr class='%s'><td>%s</td><td>%s</td><td>%d</td><td>%s</td><td>%s</td><td>%s</td><td>%s</td><td>%d</td></tr>"
    % ("top-video" if st["is_top"] else "", _esc(qid),
       _esc(_query_type_label(RESULTS[qid].get("query_type", ""))), st["occurrences"],
       str(st["events"]) if str(RESULTS[qid].get("query_type", "")).lower() == "trake" else "-",
       _esc("; ".join("#%d" % x for x in st["ranks"])),
       _esc("#%s" % st["best_rank"] if st["best_rank"] is not None else "-"),
       _esc("%.3f" % st["best_score"] if st["best_score"] is not None else "-"),
       _esc(st["video_id"]), _review_image_counts.get((str(qid), str(st["video_id"])), 0))
    for qid, stats in _QUERY_VIDEO_STATS.items() for st in stats)

# Tổng hợp toàn bộ truy vấn: số lần xuất hiện được tính từ ứng viên/chuỗi, không phải số ảnh.
_video_summary = defaultdict(lambda: {"occurrences": 0, "events": 0, "images": 0,
                                      "queries": set(), "best_rank": None,
                                      "image_names": []})
for qid, stats in _QUERY_VIDEO_STATS.items():
    for st_query in stats:
        st = _video_summary[str(st_query["video_id"])]
        st["occurrences"] += st_query["occurrences"]
        st["events"] += st_query["events"]
        st["queries"].add(str(qid))
        rank = st_query["best_rank"]
        st["best_rank"] = rank if st["best_rank"] is None else min(st["best_rank"], rank)
for item in _review_images:
    st = _video_summary[str(item["video_id"])]
    st["images"] += 1
    if item["image_name"] and item["image_name"] not in st["image_names"]:
        st["image_names"].append(item["image_name"])
_video_rows_html = "".join(
    "<tr><td>%s</td><td>%d</td><td>%s</td><td>%d</td><td>%s</td><td>%d</td><td>%s</td></tr>"
    % (_esc(vid), st["occurrences"],
       str(st["events"]) if st["events"] else "-", len(st["queries"]),
       "#%s" % st["best_rank"] if st["best_rank"] is not None else "-",
       st["images"], _esc(", ".join(st["image_names"][:8]) + (" ..." if len(st["image_names"]) > 8 else "")))
    for vid, st in sorted(_video_summary.items(), key=lambda z: (-z[1]["occurrences"], z[0])))
_review_images_html = "".join(
    "<tr><td>%s</td><td class='nowrap'>%s</td><td>%s</td><td>%s</td>"
    "<td>%s</td><td>%s</td><td>%s</td><td>%s</td>"
    "<td><a href='%s' target='_blank'><img class='thumb' loading='lazy' alt='%s' src='%s'></a></td></tr>"
    % (_esc(item["query_id"]), _esc(_query_type_label(item["query_type"])), _esc(item["video_id"]),
       _esc(str(item["rank"]) + ((" / E" + str(item["event"])) if item["event"] != "" else "")),
       _esc(item["image_name"]), _esc(item["source_image_name"]),
       _esc(str(item["frame_idx"])), _esc("%.3f" % item["pts_time"]),
       _esc(item["image"]), _esc(item["image_name"]), _esc(item["image"]))
    for item in _review_images)
(OUT / "review.html").write_text(
    "<!doctype html><html lang='vi'><head><meta charset='UTF-8'><meta name='viewport' content='width=device-width, initial-scale=1'><title>AIC - Rà soát ứng viên</title></head><body>%s"
    "<h1>Gói rà soát - %d truy vấn</h1>"
    "<p><b>%d video</b> | <b>%d ảnh</b> trong gói rà soát.</p>"
    "<h2>Tóm tắt từng truy vấn</h2><div class='scroll'><table><tr><th>mã truy vấn</th><th>loại</th><th>ảnh rà soát</th><th>ứng viên / chuỗi</th><th>số video</th><th>video xuất hiện nhiều nhất</th><th>nội dung truy vấn</th></tr>%s</table></div>"
    "<h2>Video trong từng truy vấn</h2>"
    "<p class='note'>Bảng này liệt kê toàn bộ video của từng truy vấn và sắp xếp theo số lần xuất hiện giảm dần. Dòng nền xanh là video xuất hiện nhiều nhất trong truy vấn đó; với TRAKE, mỗi chuỗi sự kiện được tính là một lần và tổng sự kiện là tổng số mốc.</p>"
    "<div class='scroll'><table><tr><th>mã truy vấn</th><th>loại</th><th>mã video</th><th>số lần xuất hiện</th><th>tổng sự kiện</th><th>các hạng</th><th>hạng tốt nhất</th><th>điểm tốt nhất</th><th>ảnh rà soát</th></tr>%s</table></div>"
    "<h2>Tổng hợp video trên toàn bộ truy vấn</h2>"
    "<div class='scroll'><table><tr><th>mã video</th><th>tổng số lần xuất hiện</th><th>tổng sự kiện</th><th>số truy vấn</th><th>hạng tốt nhất</th><th>ảnh rà soát</th><th>tên tệp ảnh</th></tr>%s</table></div>"
    "<h2>Chi tiết ảnh rà soát</h2>"
    "<div class='scroll'><table><tr><th>mã truy vấn</th><th>loại</th><th>mã video</th><th>hạng / sự kiện</th><th>tên tệp ảnh</th><th>tên ảnh nguồn</th><th>khung hình</th><th>thời gian (giây)</th><th>xem trước</th></tr>%s</table></div></body></html>"
    % (_CSS, len(_index_rows), len(_video_summary), len(_review_images), _query_overview_rows_html,
       _query_video_rows_html, _video_rows_html, _review_images_html), encoding="utf-8")
print("Trang rà soát ->", OUT / "review.html")

REVIEW_MANIFEST = {
    "created_ts": time.time(),
    "Notebook": "02_retrieve_refine_candidates_api",
    "nb01_manifest": str(ART / "artifact_manifest.json"),
    "artifacts_dir": str(ART),
    "review_package_dir": str(OUT),
    "config": CFG,
    "degraded": {
        "visual_branch": bool(VISUAL_OK),
        "siglip_reason": SIG.get("reason", ""),
        "text_embed_indices": list(TEXT_IX.keys()),
        "object_boost": bool(OBJ_BY_ENT),
        "refine_available": bool(HAS_CV2 and HAS_PIL and len(VIDEO_FILES) > 0),
        "dry_run": CFG["DRY_RUN"],
        "api_off": dict(API_OFF),
    },
    "queries": [{"query_id": qid, "query_type": r["query_type"], "profile": r["profile"],
                 "n_rows": len(r["rows"]), "n_sequences": len(r["sequences"]),
                 "n_events": len(r["structured"].get("events") or []),
                 "n_unique_videos": len(_QUERY_VIDEO_STATS.get(qid, [])),
                 "top_videos": [{"video_id": st["video_id"], "occurrences": st["occurrences"]}
                               for st in _QUERY_VIDEO_STATS.get(qid, []) if st["is_top"]],
                 "warnings": r.get("warnings", []),
                 "path": str(OUT / "candidates" / (qid + ".json"))}
                for qid, r in RESULTS.items()],
    "review_stats": {"n_videos": len(_video_summary), "n_images": len(_review_images),
                    "review_html": str(OUT / "review.html")},
    "COST": {"nb02_spent_usd": round(NB02_SPENT["usd"], 5), "ledger": ledger_read()},
    "handoff_notes": [
        "NB03 Chỉ đọc review_package + artifacts; Không gọi API, Không Build lại Index.",
        "rows[*].frame_idx là frame Cuối cùng để nộp (đã Refine nếu refined=True).",
        "TRAKE: mỗi phần từ sequences là 1 dòng CSV; FRAMES phải tăng dần và đủ N event.",
        "Q&A: Answer <= 100 ký tự; người Review có thể sửa trước Khi xuất CSV.",
        "Ưu tiên Review: toàn bộ TRAKE > Q&A cần đếm/OCR > Query confidence thấp > top5 còn lại.",
    ],
}
jdump(REVIEW_MANIFEST, OUT / "review_manifest.json")
print("\nreview manifest ->", OUT / "review_manifest.json")
print("degraded:", json.dumps(REVIEW_MANIFEST["degraded"], ensure_ascii=False))

# --- Smoke test bàn giao ---
ok = True
def chk(n, c, d=""):
    global ok
    print(("  PASS " if c else "  FAIL ") + n + (("  (" + d + ")") if d else ""))
    ok = ok and bool(c)

chk("review_manifest", (OUT / "review_manifest.json").exists())
chk("Candidates Files", len(list((OUT / "candidates").glob("*.json"))) == len(RESULTS),
    "%d Files" % len(list((OUT / "candidates").glob("*.json"))))
for qid, r in list(RESULTS.items())[:3]:
    if r["query_type"] == "trake":
        chk("%s sequences" % qid, len(r["sequences"]) > 0)
        if r["sequences"]:
            s = r["sequences"][0]
            chk("%s FRAMES==n_events" % qid,
                len(s["frames"]) == len(r["structured"]["events"]),
                "%d vs %d" % (len(s["frames"]), len(r["structured"]["events"])))
    else:
        chk("%s rows<=100" % qid, len(r["rows"]) <= 100, "%d" % len(r["rows"]))
        chk("%s frame_idx int" % qid, all(isinstance(x["frame_idx"], int) for x in r["rows"]))
chk("COST dưới CAP", ledger_read()["total_usd"] < CFG["MAX_TOTAL_COST_USD"],
    "$%.4f" % ledger_read()["total_usd"])
print("\nSMOKE:", "OK" if ok else "FAILED")
print("Tiếp theo: CELL 17 - xuất submission CSV + ZIP")

## Xuất file nộp (submission)

NB02 trước đây chỉ dừng ở review package nên Kaggle output không có file nộp. Cell dưới xuất thẳng gói nộp theo `TheLeCuocThi-DeThi/sotuyenAIC.md`:

- `submission/<query_id>.csv`, không header, UTF-8, tối đa 100 dòng.
- KIS `video_id,frame_idx` — QA `video_id,frame_idx,answer` (≤100 ký tự) — TRAKE `video_id,frame_1,...,frame_N` (đúng N event, tăng dần).
- Nén thành `/kaggle/working/submission.zip` **có** thư mục `submission/` bên trong.

Q&A: NB02 chạy `allow_vlm_verify=False` nên `answer` rỗng. Điền `/kaggle/working/qa_answers.json` dạng `{"query-p1-15-qa": "12"}` rồi chạy lại đúng cell này (không cần chạy lại pipeline).


In [ ]:
# ============================== CELL 17: EXPORT SUBMISSION (CSV + ZIP) ==============================
# Xuất đúng định dạng BTC (TheLeCuocThi-DeThi/sotuyenAIC.md):
#   submission/<query_id>.csv  -> nén thành .zip CÓ thư mục submission/ bên trong.
#   KIS   : <video_id>,<frame_idx>
#   QA    : <video_id>,<frame_idx>,<answer>           (answer <= 100 ký tự)
#   TRAKE : <video_id>,<frame_1>,...,<frame_N>        (đúng N event, tăng dần)
# Không header, UTF-8, phân cách bằng dấu phẩy, tối đa 100 dòng mỗi file.
# Tên file CSV lấy nguyên tên file đề, chỉ đổi đuôi: query-p1-1-kis.txt -> query-p1-1-kis.csv
import csv, zipfile

SUB_CFG = {
    "sub_root":  "/kaggle/working/submission_build",   # thư mục dựng gói nộp
    "zip_path":  "/kaggle/working/submission.zip",     # file nộp lên hệ thống BTC
    "zip_inner": "submission",                         # BẮT BUỘC theo thể lệ
    "max_rows":  100,               # trần cứng của BTC, không được vượt
    # Trần riêng theo loại query. Để 100 = dùng hết quota BTC.
    # Hạ xuống dưới 100 sẽ tự khoá trần điểm: nộp 20 dòng -> R@50 = R@100 = R@20
    # -> Final Score của query đó tối đa chỉ còn 0.6 thay vì 1.0.
    "max_rows_by_type": {"kis": 100, "qa": 20, "trake": 100},
    # Answer cho Q&A. NB02 mặc định allow_vlm_verify=False -> row["answer"] rỗng.
    # Ghi đè bằng file JSON {"query-p1-15-qa": "12", ...} hoặc dict dưới đây.
    # Đáp án là một chuỗi duy nhất cho cả query nên được rải vào TOÀN BỘ các dòng.
    "qa_answer_file": "/kaggle/working/qa_answers.json",
    "qa_answers": {},
    "qa_answer_fallback": "",       # answer rỗng vẫn hợp lệ về format (nhưng R-Score = 0)
    "write_empty_file": False,      # query không có candidate -> cảnh báo, không tạo file rỗng
    "lineterminator": "\n",
}

# Cell này chạy độc lập được: nếu mất RESULTS (restart kernel) thì đọc lại checkpoint.
if "RESULTS" not in globals() or not RESULTS:
    RESULTS = {}
    for _p in sorted((OUT / "candidates").glob("*.json")):
        try:
            _r = jload(_p)
            RESULTS[_r["query_id"]] = _r
        except Exception as _e:
            print("[warn] không đọc được", _p.name, _e)
    print("[info] nạp lại RESULTS từ checkpoint:", len(RESULTS), "query")

_QA_OVERRIDE = dict(SUB_CFG["qa_answers"])
_qaf = Path(SUB_CFG["qa_answer_file"])
if _qaf.exists():
    try:
        _QA_OVERRIDE.update({str(k): str(v) for k, v in jload(_qaf).items()})
        print("[info] qa_answers ghi đè:", sorted(_QA_OVERRIDE))
    except Exception as e:
        print("[warn] qa_answers.json lỗi:", e)


def _vid(v):
    """Tên file video KHÔNG có đuôi .mp4."""
    s = str(v).strip().replace("\\", "/").split("/")[-1]
    for ext in (".mp4", ".mkv", ".webm", ".avi", ".mov"):
        if s.lower().endswith(ext):
            s = s[: -len(ext)]
    return s


def _fidx(x):
    """Frame ID là số nguyên >= 0."""
    return max(0, int(round(float(x))))


def _answer(qid, row):
    a = _QA_OVERRIDE.get(qid) or (row.get("answer") or "") or SUB_CFG["qa_answer_fallback"]
    a = " ".join(str(a).split())          # bỏ xuống dòng / khoảng trắng thừa
    return a[:100]                        # trần 100 ký tự của BTC


def rows_for_query(qid, r):
    """RESULTS[qid] -> (list[list[str]], list[cảnh báo]) đúng format theo query_type."""
    qt = str(r.get("query_type", "kis")).lower()
    warn, out, seen = [], [], set()
    cap = min(int(SUB_CFG["max_rows"]),
              int((SUB_CFG.get("max_rows_by_type") or {}).get(qt, SUB_CFG["max_rows"])))

    if qt == "trake":
        n_events = len(((r.get("structured") or {}).get("events")) or []) \
                   or len(r.get("events_raw") or [])
        seqs = r.get("sequences") or []
        if not n_events and seqs:
            n_events = len(seqs[0].get("frames") or [])
            warn.append("không đọc được số event -> suy ra N=%d từ sequence đầu" % n_events)
        for s in seqs:
            fr = [_fidx(f) for f in (s.get("frames") or [])]
            if not fr:
                continue
            if n_events and len(fr) != n_events:
                warn.append("bỏ seq rank=%s: %d frame khác %d event"
                            % (s.get("rank"), len(fr), n_events))
                continue
            if any(fr[i] > fr[i + 1] for i in range(len(fr) - 1)):
                warn.append("seq rank=%s không tăng dần -> đã sort" % s.get("rank"))
                fr = sorted(fr)
            rec = [_vid(s.get("video_id"))] + [str(f) for f in fr]
            key = tuple(rec)
            if key in seen:
                continue
            seen.add(key)
            out.append(rec)
            if len(out) >= cap:
                break
    else:
        for x in (r.get("rows") or []):
            rec = [_vid(x.get("video_id")), str(_fidx(x.get("frame_idx")))]
            if qt == "qa":
                rec.append(_answer(qid, x))
            key = (rec[0], rec[1])        # trùng (video, frame) -> vô nghĩa, bỏ
            if key in seen:
                continue
            seen.add(key)
            out.append(rec)
            if len(out) >= cap:
                break
        if qt == "qa" and out and not any(rec[2] for rec in out):
            warn.append("TẤT CẢ answer đều rỗng -> Q&A sẽ được 0 điểm; "
                        "điền qa_answers.json rồi chạy lại cell này")
    return out, warn


SUB_ROOT = Path(SUB_CFG["sub_root"])
SUB_DIR = SUB_ROOT / SUB_CFG["zip_inner"]
if SUB_ROOT.exists():
    shutil.rmtree(SUB_ROOT)
SUB_DIR.mkdir(parents=True, exist_ok=True)

SUB_REPORT, n_files, n_lines = [], 0, 0
for qid in sorted(RESULTS.keys()):
    r = RESULTS[qid]
    recs, warn = rows_for_query(qid, r)
    if not recs and not SUB_CFG["write_empty_file"]:
        print("  [BỎ]   %-24s không có candidate nào. %s" % (qid, "; ".join(warn)))
        SUB_REPORT.append({"query_id": qid, "type": r.get("query_type"), "file": None,
                           "n_lines": 0, "warnings": warn + ["không xuất file"]})
        continue
    fp = SUB_DIR / (qid + ".csv")       # tên đề, chỉ đổi đuôi .txt -> .csv
    with open(fp, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter=",", quotechar='"', quoting=csv.QUOTE_MINIMAL,
                       lineterminator=SUB_CFG["lineterminator"])
        w.writerows(recs)
    n_files += 1
    n_lines += len(recs)
    SUB_REPORT.append({"query_id": qid, "type": r.get("query_type"),
                       "file": fp.name, "n_lines": len(recs), "warnings": warn})
    _cap_t = int((SUB_CFG.get("max_rows_by_type") or {}).get(
        str(r.get("query_type")).lower(), SUB_CFG["max_rows"]))
    _note = "  [chặn %d/100]" % _cap_t if _cap_t < SUB_CFG["max_rows"] else ""
    print("  [OK]   %-24s %-5s %3d dòng -> %s%s%s"
          % (qid, r.get("query_type"), len(recs), fp.name, _note,
             ("  | " + "; ".join(warn[:2])) if warn else ""))

# ---------- Validate lại chính file đã ghi (đọc ngược bằng csv reader) ----------
val_err = []
for item in SUB_REPORT:
    if not item["file"]:
        continue
    fp = SUB_DIR / item["file"]
    qt = str(item["type"]).lower()
    with open(fp, "r", encoding="utf-8", newline="") as f:
        rows = [row for row in csv.reader(f) if row]
    if len(rows) > SUB_CFG["max_rows"]:
        val_err.append("%s: %d dòng > 100" % (fp.name, len(rows)))
    ncols = {len(row) for row in rows}
    if qt == "kis" and ncols - {2}:
        val_err.append("%s: KIS phải có đúng 2 cột, thấy %s" % (fp.name, sorted(ncols)))
    if qt == "qa" and ncols - {3}:
        val_err.append("%s: QA phải có đúng 3 cột, thấy %s" % (fp.name, sorted(ncols)))
    if qt == "trake" and len(ncols) > 1:
        val_err.append("%s: TRAKE số frame không đồng nhất %s" % (fp.name, sorted(ncols)))
    for i, row in enumerate(rows, 1):
        if not re.fullmatch(r"[A-Za-z0-9_\-]+", row[0]):
            val_err.append("%s dòng %d: video_id là '%s'" % (fp.name, i, row[0]))
            break
        frame_cells = row[1:2] if qt == "qa" else row[1:]
        bad = [c for c in frame_cells if not re.fullmatch(r"\d+", c)]
        if bad:
            val_err.append("%s dòng %d: frame_idx không phải số nguyên %s" % (fp.name, i, bad))
            break
        if qt == "qa" and len(row[2]) > 100:
            val_err.append("%s dòng %d: answer dài %d ký tự > 100" % (fp.name, i, len(row[2])))
            break
val_ok = not val_err

# ---------- Đối chiếu với danh sách query-*.txt mà BTC giao ----------
# Thiếu file = query đó chắc chắn 0 điểm. Trường hợp nguy hiểm nhất là pipeline
# "break" giữa chừng (hết budget / RuntimeError): query phía sau không vào RESULTS
# nên không có cả dòng [BỎ], im lặng biến mất khỏi zip.
EXPECTED_Q = []
try:
    EXPECTED_Q = [x.stem for x in _query_files(QDIR)]
except Exception as e:
    print("[warn] không đọc lại được danh sách query (%s) -> bỏ qua bước đối chiếu"
          % type(e).__name__)
_written = {x["query_id"] for x in SUB_REPORT if x["file"]}
MISSING_Q = [q for q in EXPECTED_Q if q not in _written]
EXTRA_Q = sorted(_written - set(EXPECTED_Q)) if EXPECTED_Q else []

# ---------- Nén: zip PHẢI chứa thư mục submission/ ----------
ZIP = Path(SUB_CFG["zip_path"])
if ZIP.exists():
    ZIP.unlink()
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in sorted(SUB_DIR.glob("*.csv")):
        zf.write(fp, arcname="%s/%s" % (SUB_CFG["zip_inner"], fp.name))

jdump({"created_ts": time.time(), "n_files": n_files, "n_lines": n_lines,
       "zip": str(ZIP), "valid": val_ok and not MISSING_Q,
       "errors": val_err, "files": SUB_REPORT,
       "n_expected": len(EXPECTED_Q), "missing_queries": MISSING_Q,
       "unexpected_queries": EXTRA_Q},
      OUT / "submission_report.json")

print("\n" + "=" * 70)
print("SUBMISSION -> %s" % SUB_DIR)
print("ZIP        -> %s  (%.1f KB)" % (ZIP, ZIP.stat().st_size / 1024.0))
print("số file: %d | tổng số dòng: %d" % (n_files, n_lines))
with zipfile.ZipFile(ZIP) as zf:
    _names = zf.namelist()
print("zip chứa:", _names[:5], ("... +%d file" % (len(_names) - 5)) if len(_names) > 5 else "")
if val_err:
    print("\nLỖI FORMAT (%d):" % len(val_err))
    for e in val_err[:20]:
        print("  -", e)
else:
    print("kiểm tra format: PASS (đúng thể lệ sotuyenAIC.md)")
if EXPECTED_Q:
    print("đối chiếu đề thi: %d/%d query có file CSV" % (len(_written), len(EXPECTED_Q)))
    if MISSING_Q:
        print("  !! THIẾU FILE (%d) -> những query này chắc chắn 0 điểm:" % len(MISSING_Q))
        for q in MISSING_Q:
            _r = RESULTS.get(q)
            print("     - %-26s %s" % (q, "không có candidate" if _r else
                  "KHÔNG CHẠY (pipeline dừng sớm? xem log CELL 14)"))
    if EXTRA_Q:
        print("  !! CSV thừa, không có query-*.txt tương ứng:", EXTRA_Q)
    if not MISSING_Q and not EXTRA_Q:
        print("  đủ file, tên khớp 1-1 với query-*.txt")
_qa_blank = [x["query_id"] for x in SUB_REPORT
             if str(x["type"]).lower() == "qa"
             and any("answer đều rỗng" in w for w in x["warnings"])]
if _qa_blank:
    print("Q&A chưa có answer:", _qa_blank,
          "-> tạo /kaggle/working/qa_answers.json rồi chạy lại CELL 17")
print("=" * 70)
print("Tải file: Kaggle -> Output -> submission.zip (nộp trực tiếp lên hệ thống BTC)")
